In [1]:
!nvidia-smi

Tue Apr 28 11:56:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/andypinxinliu/GestureLSM
%cd GestureLSM

Cloning into 'GestureLSM'...
remote: Enumerating objects: 353, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 353 (delta 84), reused 69 (delta 69), pack-reused 248 (from 1)
Receiving objects: 100% (353/353), 82.74 MiB | 24.92 MiB/s, done.
Resolving deltas: 100% (155/155), done.
/content/GestureLSM


In [3]:
# ffmpeg needed by whisperx/librosa, OpenGL for pyrender
!apt-get install -y ffmpeg libegl1 libegl-dev libgl1-mesa-dev libglib2.0-0 -q

Reading package lists...
Building dependency tree...
Reading state information...
libegl1 is already the newest version (1.4.0-1).
libglib2.0-0 is already the newest version (2.72.4-0ubuntu2.9).
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
The following additional packages will be installed:
  libgl-dev libgles-dev libgles1 libglvnd-core-dev libglvnd-dev libglx-dev
  libopengl-dev
The following NEW packages will be installed:
  libegl-dev libgl-dev libgl1-mesa-dev libgles-dev libgles1 libglvnd-core-dev
  libglvnd-dev libglx-dev libopengl-dev
0 upgraded, 9 newly installed, 0 to remove and 42 not upgraded.
Need to get 221 kB of archives.
After this operation, 2,577 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libglx-dev amd64 1.4.0-1 [14.1 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libgl-dev amd64 1.4.0-1 [101 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/main amd64 libegl-dev amd64 1.4.0-1 [18.0

In [4]:
# Colab typically has CUDA 12.x — use cu121 build
!pip install torch torchvision torchaudio

In [5]:
# 'fasttest' in requirements.txt is a typo — install fasttext-wheel instead
!pip install -r /content/GestureLSM/requirements.txt --ignore-requires-python -q || true
!pip install fasttext-wheel -q   # fixes the fasttest typo
!pip install pgvector biopython -q

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 7.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 72.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 101.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 24.8 MB/s et

In [6]:
!pip install whisperx -q
!pip install textgrid -q   # for writing .TextGrid files WhisperX -> TextGrid

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.0/39.0 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.7/893.7 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.9/887.9 MB 1.9 MB/s eta 0:00:00
   ━

In [7]:
%cd /content/GestureLSM
%ls

/content/GestureLSM
beat-new.png                mean_std/                   static/
configs/                    models/                     test.py
configs_new/                optimizers/                 test_sample_t.py
dataloaders/                preprocess/                 trainer/
demo/                       README.md                   train.py
demo.py                     requirements.txt            train_rvq.sh
diffuser_rvqvae_trainer.py  rvq_beatx_test.py           utils/
index.html                  rvq_beatx_train.py          weights/
LICENSE                     shortcut_rvqvae_trainer.py


In [8]:
# Option A: HuggingFace (most reliable)
!huggingface-cli download pliu23/GestureLSM --local-dir ./ckpt

# SMPL models
!pip install gdown -q
!gdown "https://drive.google.com/drive/folders/1MCks7CMNBtAzU2XihYezNmiGT_6pWex8?usp=drive_link" \
    -O ./datasets/hub --folder

⚠️  Warning: 'huggingface-cli download' is deprecated. Use 'hf download' instead.
Fetching 13 files:   0% 0/13 [00:00<?, ?it/s]Downloading '.gitattributes' to 'ckpt/.cache/huggingface/download/wPaCkH-WbT7GsmxMKKrNZTV4nSM=.a6344aac8c09253b3b630fb776ae94478aa0275b.incomplete'

README.md: 2.96kB [00:00, 13.6MB/s]
Download complete. Moving file to ckpt/README.md

.gitattributes: 1.52kB [00:00, 8.61MB/s]
Download complete. Moving file to ckpt/.gitattributes
Fetching 13 files:   8% 1/13 [00:00<00:02,  5.65it/s]
hand_all_speakers.pth:   0% 0.00/75.6M [00:00<?, ?B/s]Downloading 'net_300000_upper.pth' to 'ckpt/.cache/huggingface/download/G4PMGi4q8tSEGW_B7e74uw4OAig=.12c5f88559c46a1a2eb7d148e3fc7608d0dab65b78304c83944dadc9ddf06d8f.incomplete'



net_300000_lower.pth:   0% 0.00/74.1M [00:00<?, ?B/s]

net_300000_hands.pth:   0% 0.00/75.6M [00:00<?, ?B/s]



all_speakers_denoiser.bin:   0% 0.00/156M [00:00<?, ?B/s]




net_300000_face.pth:   0% 0.00/74.6M [00:00<?, ?B/s]Downloading 'new_540_diffusi

In [9]:
%%writefile demo2.py
import os
import signal
import time
import csv
import sys
import warnings
import random
import gradio as gr
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
import torch.multiprocessing as mp
import numpy as np
import time
import pprint
from loguru import logger
import smplx
from torch.utils.tensorboard import SummaryWriter
import wandb
import matplotlib.pyplot as plt
from utils import config, logger_tools, other_tools_hf, metric, data_transfer, other_tools
from utils.joints import upper_body_mask, hands_body_mask, lower_body_mask
from dataloaders import data_tools
from dataloaders.build_vocab import Vocab
from optimizers.optim_factory import create_optimizer
from optimizers.scheduler_factory import create_scheduler
from optimizers.loss_factory import get_loss_func
from dataloaders.data_tools import joints_list
from utils import rotation_conversions as rc
import soundfile as sf
import librosa
# ── WhisperX replaces both the HuggingFace Whisper pipeline and MFA ──────────
import whisperx
# ─────────────────────────────────────────────────────────────────────────────
from models.vq.model import RVQVAE

device = "cuda" if torch.cuda.is_available() else "cpu"

import platform
if platform.system() == "Linux":
    os.environ['PYOPENGL_PLATFORM'] = 'egl'

# ── Load WhisperX model once at module level (mirrors original `pipe` pattern) ─
#    "tiny.en" keeps it fast; swap for "base.en" or "small.en" for better accuracy.
#    compute_type="float16" requires GPU; falls back to "int8" on CPU.
_compute_type = "float16" if torch.cuda.is_available() else "int8"
whisperx_asr_model = whisperx.load_model(
    "base.en",          # change to "small.en" for better accuracy at cost of speed
    device,
    compute_type=_compute_type,
    language="en",
)
# ─────────────────────────────────────────────────────────────────────────────

debug = False


# ── TextGrid helpers ──────────────────────────────────────────────────────────

def _whisperx_to_textgrid(word_segments, output_path, duration):
    """
    Convert WhisperX word-level segments → Praat TextGrid file.

    WhisperX gives:
        [{"word": "hello", "start": 0.04, "end": 0.44, "score": 0.97}, ...]

    TextGrid IntervalTier expected by GestureLSM dataloaders (same schema as
    what MFA produces):
        File type = "ooTextFile"
        Object class = "TextGrid"
        ...
        item [1]:  class = "IntervalTier"  name = "words"
            intervals [N]: xmin / xmax / text
    """
    # Filter out segments without timing (WhisperX occasionally skips a word)
    segs = [s for s in word_segments if "start" in s and "end" in s]

    # Fill any gap at the start with a silent interval
    intervals = []
    cursor = 0.0
    for seg in segs:
        if seg["start"] > cursor + 1e-6:
            intervals.append({"xmin": cursor, "xmax": seg["start"], "text": ""})
        intervals.append({"xmin": seg["start"], "xmax": seg["end"], "text": seg["word"].strip()})
        cursor = seg["end"]
    # Fill tail silence
    if cursor < duration - 1e-6:
        intervals.append({"xmin": cursor, "xmax": duration, "text": ""})

    with open(output_path, "w", encoding="utf-8") as f:
        f.write('File type = "ooTextFile"\n')
        f.write('Object class = "TextGrid"\n\n')
        f.write(f'xmin = 0\n')
        f.write(f'xmax = {duration:.6f}\n')
        f.write('tiers? <exists>\n')
        f.write('size = 2\n')
        f.write('item []:\n')

        # Tier 1 – words  (what MFA writes; dataloaders read this)
        f.write('    item [1]:\n')
        f.write('        class = "IntervalTier"\n')
        f.write('        name = "words"\n')
        f.write(f'        xmin = 0\n')
        f.write(f'        xmax = {duration:.6f}\n')
        f.write(f'        intervals: size = {len(intervals)}\n')
        for i, iv in enumerate(intervals, 1):
            f.write(f'        intervals [{i}]:\n')
            f.write(f'            xmin = {iv["xmin"]:.6f}\n')
            f.write(f'            xmax = {iv["xmax"]:.6f}\n')
            f.write(f'            text = "{iv["text"]}"\n')

        # Tier 2 – phones  (MFA also writes this; keep it empty so dataloaders
        #           that peek at tier count don't crash)
        f.write('    item [2]:\n')
        f.write('        class = "IntervalTier"\n')
        f.write('        name = "phones"\n')
        f.write(f'        xmin = 0\n')
        f.write(f'        xmax = {duration:.6f}\n')
        f.write('        intervals: size = 1\n')
        f.write('        intervals [1]:\n')
        f.write('            xmin = 0\n')
        f.write(f'            xmax = {duration:.6f}\n')
        f.write('            text = ""\n')


def _run_whisperx_alignment(audio_path, tmp_dir, textgrid_path):
    """
    Full WhisperX pipeline:
        1. ASR  →  transcript + rough word timestamps
        2. Forced alignment  →  tight word timestamps
        3. Write .lab file (plain text transcript, same as MFA input)
        4. Write .TextGrid  (Praat format, same as MFA output)

    Returns the transcript string (in case caller needs it).
    """
    # ── 1. ASR ────────────────────────────────────────────────────────────────
    audio_array = whisperx.load_audio(audio_path)
    result = whisperx_asr_model.transcribe(audio_array, batch_size=8)
    # result["segments"] is a list of {text, start, end, words: [...]}

    transcript = " ".join(seg["text"].strip() for seg in result["segments"])
    lab_path = os.path.join(tmp_dir, "tmp.lab")
    with open(lab_path, "w", encoding="utf-8") as f:
        f.write(transcript)

    # ── 2. Forced alignment ───────────────────────────────────────────────────
    #    whisperx downloads a small wav2vec2 alignment model on first run.
    #    Subsequent runs use the cached version (~300 MB in ~/.cache).
    align_model, align_metadata = whisperx.load_align_model(
        language_code="en", device=device
    )
    result_aligned = whisperx.align(
        result["segments"],
        align_model,
        align_metadata,
        audio_array,
        device,
        return_char_alignments=False,
    )
    # result_aligned["word_segments"]: list of {"word", "start", "end", "score"}

    # ── 3. Write TextGrid ─────────────────────────────────────────────────────
    duration = len(audio_array) / 16000.0   # whisperx always uses 16 kHz
    _whisperx_to_textgrid(result_aligned["word_segments"], textgrid_path, duration)

    logger.info(
        f"WhisperX alignment done. "
        f"{len(result_aligned['word_segments'])} words → {textgrid_path}"
    )
    return transcript


# ─────────────────────────────────────────────────────────────────────────────

class BaseTrainer(object):
    def __init__(self, args, cfg, ap):

        hf_dir = "hf"
        time_local = time.localtime()
        time_name_expend = "%02d%02d_%02d%02d%02d_" % (
            time_local[1], time_local[2], time_local[3], time_local[4], time_local[5]
        )
        self.time_name_expend = time_name_expend
        tmp_dir = args.out_path + "custom/" + time_name_expend + hf_dir
        if not os.path.exists(tmp_dir + "/"):
            os.makedirs(tmp_dir + "/")

        self.audio_path = tmp_dir + "/tmp.wav"
        sf.write(self.audio_path, ap[1], ap[0])

        audio, ssr = librosa.load(self.audio_path, sr=args.audio_sr)

        self.textgrid_path = tmp_dir + "/tmp.TextGrid"

        # ── CHANGED: WhisperX replaces Whisper pipeline + MFA subprocess ──────
        if not debug:
            _run_whisperx_alignment(self.audio_path, tmp_dir, self.textgrid_path)
        # ──────────────────────────────────────────────────────────────────────

        ap = (ssr, audio)
        self.args = args
        self.rank = 0

        args.textgrid_file_path = self.textgrid_path
        args.audio_file_path = self.audio_path

        self.rank = 0

        self.checkpoint_path = tmp_dir
        args.tmp_dir = tmp_dir
        if self.rank == 0:
            self.test_data = __import__(
                f"dataloaders.{args.dataset}", fromlist=["something"]
            ).CustomDataset(args, "test")
            self.test_loader = torch.utils.data.DataLoader(
                self.test_data,
                batch_size=1,
                shuffle=False,
                num_workers=args.loader_workers,
                drop_last=False,
            )
        logger.info(f"Init test dataloader success")
        model_module = __import__(f"models.{cfg.model.model_name}", fromlist=["something"])

        self.model = torch.nn.DataParallel(
            getattr(model_module, cfg.model.g_name)(cfg), args.gpus
        ).cuda()

        if self.rank == 0:
            logger.info(self.model)
            logger.info(f"init {cfg.model.g_name} success")

        self.smplx = smplx.create(
            self.args.data_path_1 + "smplx_models/",
            model_type='smplx',
            gender='NEUTRAL_2020',
            use_face_contour=False,
            num_betas=300,
            num_expression_coeffs=100,
            ext='npz',
            use_pca=False,
        ).to(self.rank).eval()

        self.args = args
        self.ori_joint_list = joints_list[self.args.ori_joints]
        self.tar_joint_list_face = joints_list["beat_smplx_face"]
        self.tar_joint_list_upper = joints_list["beat_smplx_upper"]
        self.tar_joint_list_hands = joints_list["beat_smplx_hands"]
        self.tar_joint_list_lower = joints_list["beat_smplx_lower"]

        self.joint_mask_face = np.zeros(len(list(self.ori_joint_list.keys())) * 3)
        self.joints = 55
        for joint_name in self.tar_joint_list_face:
            self.joint_mask_face[
                self.ori_joint_list[joint_name][1] - self.ori_joint_list[joint_name][0]:
                self.ori_joint_list[joint_name][1]
            ] = 1
        self.joint_mask_upper = np.zeros(len(list(self.ori_joint_list.keys())) * 3)
        for joint_name in self.tar_joint_list_upper:
            self.joint_mask_upper[
                self.ori_joint_list[joint_name][1] - self.ori_joint_list[joint_name][0]:
                self.ori_joint_list[joint_name][1]
            ] = 1
        self.joint_mask_hands = np.zeros(len(list(self.ori_joint_list.keys())) * 3)
        for joint_name in self.tar_joint_list_hands:
            self.joint_mask_hands[
                self.ori_joint_list[joint_name][1] - self.ori_joint_list[joint_name][0]:
                self.ori_joint_list[joint_name][1]
            ] = 1
        self.joint_mask_lower = np.zeros(len(list(self.ori_joint_list.keys())) * 3)
        for joint_name in self.tar_joint_list_lower:
            self.joint_mask_lower[
                self.ori_joint_list[joint_name][1] - self.ori_joint_list[joint_name][0]:
                self.ori_joint_list[joint_name][1]
            ] = 1

        self.tracker = other_tools.EpochTracker(
            ["fid", "l1div", "bc", "rec", "trans", "vel", "transv", 'dis', 'gen',
             'acc', 'transa', 'exp', 'lvd', 'mse', "cls", "rec_face", "latent",
             "cls_full", "cls_self", "cls_word", "latent_word", "latent_self",
             "predict_x0_loss"],
            [False, True, True, False, False, False, False, False, False, False,
             False, False, False, False, False, False, False, False, False, False,
             False, False, False]
        )

        ##### VQ-VAE models #####
        vq_model_module = __import__("models.motion_representation", fromlist=["something"])
        self.vq_model_face = self._create_face_vq_model(vq_model_module)

        self.vq_models = self._create_body_vq_models()

        self.vq_model_face.eval().to(self.rank)
        for model in self.vq_models.values():
            model.eval().to(self.rank)
        self.vq_model_upper, self.vq_model_hands, self.vq_model_lower = self.vq_models.values()
        self.vqvae_latent_scale = self.args.vqvae_latent_scale

        self.args.vae_length = 240

        ##### Loss functions #####
        self.reclatent_loss = nn.MSELoss().to(self.rank)
        self.vel_loss = torch.nn.L1Loss(reduction='mean').to(self.rank)

        ##### Normalization #####
        self.use_trans = self.args.use_trans
        self.mean = np.load(args.mean_pose_path)
        self.std = np.load(args.std_pose_path)

        for part in ['upper', 'hands', 'lower']:
            mask = globals()[f'{part}_body_mask']
            setattr(self, f'mean_{part}', torch.from_numpy(self.mean[mask]).cuda())
            setattr(self, f'std_{part}', torch.from_numpy(self.std[mask]).cuda())

        if self.args.use_trans:
            self.trans_mean = torch.from_numpy(np.load(self.args.mean_trans_path)).cuda()
            self.trans_std = torch.from_numpy(np.load(self.args.std_trans_path)).cuda()

    def _create_face_vq_model(self, module):
        """Create and initialize face VQ model."""
        self.args.vae_layer = 2
        self.args.vae_length = 256
        self.args.vae_test_dim = 106
        model = getattr(module, "VQVAEConvZero")(self.args).to(self.rank)
        other_tools.load_checkpoints(
            model, "./datasets/hub/pretrained_vq/face_vertex_1layer_790.bin",
            self.args.e_name
        )
        return model

    def _create_body_vq_models(self):
        """Create VQ-VAE models for body parts."""
        vq_configs = {
            'upper': {'dim_pose': 78},
            'hands': {'dim_pose': 180},
            'lower': {'dim_pose': 54 if not self.args.use_trans else 57}
        }
        vq_models = {}
        for part, cfg in vq_configs.items():
            model = self._create_rvqvae_model(cfg['dim_pose'], part)
            vq_models[part] = model
        return vq_models

    def _create_rvqvae_model(self, dim_pose: int, body_part: str) -> RVQVAE:
        """Create a single RVQVAE model with specified configuration."""
        args = self.args
        model = RVQVAE(
            args, dim_pose, args.nb_code, args.code_dim, args.code_dim,
            args.down_t, args.stride_t, args.width, args.depth,
            args.dilation_growth_rate, args.vq_act, args.vq_norm
        )
        checkpoint_path = getattr(args, f'vqvae_{body_part}_path')
        model.load_state_dict(torch.load(checkpoint_path)['net'])
        return model

    def inverse_selection(self, filtered_t, selection_array, n):
        original_shape_t = np.zeros((n, selection_array.size))
        selected_indices = np.where(selection_array == 1)[0]
        for i in range(n):
            original_shape_t[i, selected_indices] = filtered_t[i]
        return original_shape_t

    def inverse_selection_tensor(self, filtered_t, selection_array, n):
        selection_array = torch.from_numpy(selection_array).cuda()
        original_shape_t = torch.zeros((n, 165)).cuda()
        selected_indices = torch.where(selection_array == 1)[0]
        for i in range(n):
            original_shape_t[i, selected_indices] = filtered_t[i]
        return original_shape_t

    def _load_data(self, dict_data):
        tar_pose_raw = dict_data["pose"]
        tar_pose = tar_pose_raw[:, :, :165].to(self.rank)
        tar_contact = tar_pose_raw[:, :, 165:169].to(self.rank)
        tar_trans = dict_data["trans"].to(self.rank)
        tar_trans_v = dict_data["trans_v"].to(self.rank)
        tar_exps = dict_data["facial"].to(self.rank)
        in_audio = dict_data["audio"].to(self.rank)
        if 'wavlm' in dict_data:
            wavlm = dict_data["wavlm"].to(self.rank)
        else:
            wavlm = None
        in_word = dict_data["word"].to(self.rank)
        tar_beta = dict_data["beta"].to(self.rank)
        tar_id = dict_data["id"].to(self.rank).long()
        bs, n, j = tar_pose.shape[0], tar_pose.shape[1], self.joints

        tar_pose_hands = tar_pose[:, :, 25 * 3:55 * 3]
        tar_pose_hands = rc.axis_angle_to_matrix(tar_pose_hands.reshape(bs, n, 30, 3))
        tar_pose_hands = rc.matrix_to_rotation_6d(tar_pose_hands).reshape(bs, n, 30 * 6)

        tar_pose_upper = tar_pose[:, :, self.joint_mask_upper.astype(bool)]
        tar_pose_upper = rc.axis_angle_to_matrix(tar_pose_upper.reshape(bs, n, 13, 3))
        tar_pose_upper = rc.matrix_to_rotation_6d(tar_pose_upper).reshape(bs, n, 13 * 6)

        tar_pose_leg = tar_pose[:, :, self.joint_mask_lower.astype(bool)]
        tar_pose_leg = rc.axis_angle_to_matrix(tar_pose_leg.reshape(bs, n, 9, 3))
        tar_pose_leg = rc.matrix_to_rotation_6d(tar_pose_leg).reshape(bs, n, 9 * 6)

        tar_pose_lower = tar_pose_leg

        if self.args.pose_norm:
            tar_pose_upper = (tar_pose_upper - self.mean_upper) / self.std_upper
            tar_pose_hands = (tar_pose_hands - self.mean_hands) / self.std_hands
            tar_pose_lower = (tar_pose_lower - self.mean_lower) / self.std_lower

        if self.use_trans:
            tar_trans_v = (tar_trans_v - self.trans_mean) / self.trans_std
            tar_pose_lower = torch.cat([tar_pose_lower, tar_trans_v], dim=-1)

        latent_upper_top = self.vq_model_upper.map2latent(tar_pose_upper)
        latent_hands_top = self.vq_model_hands.map2latent(tar_pose_hands)
        latent_lower_top = self.vq_model_lower.map2latent(tar_pose_lower)

        latent_in = torch.cat(
            [latent_upper_top, latent_hands_top, latent_lower_top], dim=2
        ) / self.args.vqvae_latent_scale

        style_feature = None

        return {
            "in_audio": in_audio,
            "wavlm": wavlm,
            "in_word": in_word,
            "tar_trans": tar_trans,
            "tar_exps": tar_exps,
            "tar_beta": tar_beta,
            "tar_pose": tar_pose,
            "latent_in": latent_in,
            "tar_id": tar_id,
            "tar_contact": tar_contact,
            "style_feature": style_feature,
        }

    def _g_test(self, loaded_data):

        mode = 'test'
        bs, n, j = loaded_data["tar_pose"].shape[0], loaded_data["tar_pose"].shape[1], self.joints
        tar_pose = loaded_data["tar_pose"]
        tar_beta = loaded_data["tar_beta"]
        tar_exps = loaded_data["tar_exps"]
        tar_contact = loaded_data["tar_contact"]
        tar_trans = loaded_data["tar_trans"]
        in_word = loaded_data["in_word"]
        in_audio = loaded_data["in_audio"]
        in_x0 = loaded_data['latent_in']
        in_seed = loaded_data['latent_in']

        remain = n % 8
        if remain != 0:
            tar_pose = tar_pose[:, :-remain, :]
            tar_beta = tar_beta[:, :-remain, :]
            tar_trans = tar_trans[:, :-remain, :]
            in_word = in_word[:, :-remain]
            tar_exps = tar_exps[:, :-remain, :]
            tar_contact = tar_contact[:, :-remain, :]
            in_x0 = in_x0[:, :in_x0.shape[1] - (remain // self.args.vqvae_squeeze_scale), :]
            in_seed = in_seed[:, :in_x0.shape[1] - (remain // self.args.vqvae_squeeze_scale), :]
            n = n - remain

        tar_pose_jaw = tar_pose[:, :, 66:69]
        tar_pose_jaw = rc.axis_angle_to_matrix(tar_pose_jaw.reshape(bs, n, 1, 3))
        tar_pose_jaw = rc.matrix_to_rotation_6d(tar_pose_jaw).reshape(bs, n, 1 * 6)
        tar_pose_face = torch.cat([tar_pose_jaw, tar_exps], dim=2)

        tar_pose_hands = tar_pose[:, :, 25 * 3:55 * 3]
        tar_pose_hands = rc.axis_angle_to_matrix(tar_pose_hands.reshape(bs, n, 30, 3))
        tar_pose_hands = rc.matrix_to_rotation_6d(tar_pose_hands).reshape(bs, n, 30 * 6)

        tar_pose_upper = tar_pose[:, :, self.joint_mask_upper.astype(bool)]
        tar_pose_upper = rc.axis_angle_to_matrix(tar_pose_upper.reshape(bs, n, 13, 3))
        tar_pose_upper = rc.matrix_to_rotation_6d(tar_pose_upper).reshape(bs, n, 13 * 6)

        tar_pose_leg = tar_pose[:, :, self.joint_mask_lower.astype(bool)]
        tar_pose_leg = rc.axis_angle_to_matrix(tar_pose_leg.reshape(bs, n, 9, 3))
        tar_pose_leg = rc.matrix_to_rotation_6d(tar_pose_leg).reshape(bs, n, 9 * 6)
        tar_pose_lower = torch.cat([tar_pose_leg, tar_trans, tar_contact], dim=2)

        tar_pose_6d = rc.axis_angle_to_matrix(tar_pose.reshape(bs, n, 55, 3))
        tar_pose_6d = rc.matrix_to_rotation_6d(tar_pose_6d).reshape(bs, n, 55 * 6)
        latent_all = torch.cat([tar_pose_6d, tar_trans, tar_contact], dim=-1)

        rec_all_face = []
        rec_all_upper = []
        rec_all_lower = []
        rec_all_hands = []
        vqvae_squeeze_scale = self.args.vqvae_squeeze_scale
        roundt = (n - self.args.pre_frames * vqvae_squeeze_scale) // (
            self.args.pose_length - self.args.pre_frames * vqvae_squeeze_scale
        )
        remain = (n - self.args.pre_frames * vqvae_squeeze_scale) % (
            self.args.pose_length - self.args.pre_frames * vqvae_squeeze_scale
        )
        round_l = self.args.pose_length - self.args.pre_frames * vqvae_squeeze_scale

        for i in range(0, roundt):
            in_word_tmp = in_word[
                :, i * round_l:(i + 1) * round_l + self.args.pre_frames * vqvae_squeeze_scale
            ]
            in_audio_tmp = in_audio[
                :,
                i * (16000 // 30 * round_l):
                (i + 1) * (16000 // 30 * round_l) + 16000 // 30 * self.args.pre_frames * vqvae_squeeze_scale
            ]
            in_id_tmp = loaded_data['tar_id'][
                :, i * round_l:(i + 1) * round_l + self.args.pre_frames
            ]
            in_seed_tmp = in_seed[
                :,
                i * round_l // vqvae_squeeze_scale:
                (i + 1) * round_l // vqvae_squeeze_scale + self.args.pre_frames
            ]
            in_x0_tmp = in_x0[
                :,
                i * round_l // vqvae_squeeze_scale:
                (i + 1) * round_l // vqvae_squeeze_scale + self.args.pre_frames
            ]
            mask_val = torch.ones(
                bs, self.args.pose_length, self.args.pose_dims + 3 + 4
            ).float().cuda()
            mask_val[:, :self.args.pre_frames, :] = 0.0

            if i == 0:
                in_seed_tmp = in_seed_tmp[:, :self.args.pre_frames, :]
            else:
                in_seed_tmp = last_sample[:, -self.args.pre_frames:, :]

            cond_ = {'y': {}}
            cond_['y']['audio'] = in_audio_tmp
            cond_['y']['word'] = in_word_tmp
            cond_['y']['id'] = in_id_tmp
            cond_['y']['seed'] = in_seed_tmp
            cond_['y']['mask'] = (
                torch.zeros([self.args.batch_size, 1, 1, self.args.pose_length]) < 1
            ).cuda()
            cond_['y']['style_feature'] = torch.zeros([bs, 512]).cuda()

            shape_ = (bs, 3 * 128, 1, 32)
            sample = self.model(cond_)['latents']
            sample = sample.squeeze().permute(1, 0).unsqueeze(0)

            last_sample = sample.clone()

            rec_latent_upper = sample[..., :128]
            rec_latent_hands = sample[..., 128:2 * 128]
            rec_latent_lower = sample[..., 2 * 128:]

            if i == 0:
                rec_all_upper.append(rec_latent_upper)
                rec_all_hands.append(rec_latent_hands)
                rec_all_lower.append(rec_latent_lower)
            else:
                rec_all_upper.append(rec_latent_upper[:, self.args.pre_frames:])
                rec_all_hands.append(rec_latent_hands[:, self.args.pre_frames:])
                rec_all_lower.append(rec_latent_lower[:, self.args.pre_frames:])

        rec_all_upper = torch.cat(rec_all_upper, dim=1) * self.vqvae_latent_scale
        rec_all_hands = torch.cat(rec_all_hands, dim=1) * self.vqvae_latent_scale
        rec_all_lower = torch.cat(rec_all_lower, dim=1) * self.vqvae_latent_scale

        rec_upper = self.vq_model_upper.latent2origin(rec_all_upper)[0]
        rec_hands = self.vq_model_hands.latent2origin(rec_all_hands)[0]
        rec_lower = self.vq_model_lower.latent2origin(rec_all_lower)[0]

        if self.use_trans:
            rec_trans_v = rec_lower[..., -3:]
            rec_trans_v = rec_trans_v * self.trans_std + self.trans_mean
            rec_trans = torch.zeros_like(rec_trans_v)
            rec_trans = torch.cumsum(rec_trans_v, dim=-2)
            rec_trans[..., 1] = rec_trans_v[..., 1]
            rec_lower = rec_lower[..., :-3]

        if self.args.pose_norm:
            rec_upper = rec_upper * self.std_upper + self.mean_upper
            rec_hands = rec_hands * self.std_hands + self.mean_hands
            rec_lower = rec_lower * self.std_lower + self.mean_lower

        n = n - remain
        tar_pose = tar_pose[:, :n, :]
        tar_exps = tar_exps[:, :n, :]
        tar_trans = tar_trans[:, :n, :]
        tar_beta = tar_beta[:, :n, :]

        rec_exps = tar_exps
        rec_pose_legs = rec_lower[:, :, :54]
        bs, n = rec_pose_legs.shape[0], rec_pose_legs.shape[1]
        rec_pose_upper = rec_upper.reshape(bs, n, 13, 6)
        rec_pose_upper = rc.rotation_6d_to_matrix(rec_pose_upper)
        rec_pose_upper = rc.matrix_to_axis_angle(rec_pose_upper).reshape(bs * n, 13 * 3)
        rec_pose_upper_recover = self.inverse_selection_tensor(
            rec_pose_upper, self.joint_mask_upper, bs * n
        )
        rec_pose_lower = rec_pose_legs.reshape(bs, n, 9, 6)
        rec_pose_lower = rc.rotation_6d_to_matrix(rec_pose_lower)
        rec_lower2global = rc.matrix_to_rotation_6d(rec_pose_lower.clone()).reshape(bs, n, 9 * 6)
        rec_pose_lower = rc.matrix_to_axis_angle(rec_pose_lower).reshape(bs * n, 9 * 3)
        rec_pose_lower_recover = self.inverse_selection_tensor(
            rec_pose_lower, self.joint_mask_lower, bs * n
        )
        rec_pose_hands = rec_hands.reshape(bs, n, 30, 6)
        rec_pose_hands = rc.rotation_6d_to_matrix(rec_pose_hands)
        rec_pose_hands = rc.matrix_to_axis_angle(rec_pose_hands).reshape(bs * n, 30 * 3)
        rec_pose_hands_recover = self.inverse_selection_tensor(
            rec_pose_hands, self.joint_mask_hands, bs * n
        )
        rec_pose = rec_pose_upper_recover + rec_pose_lower_recover + rec_pose_hands_recover
        rec_pose[:, 66:69] = tar_pose.reshape(bs * n, 55 * 3)[:, 66:69]

        rec_pose = rc.axis_angle_to_matrix(rec_pose.reshape(bs * n, j, 3))
        rec_pose = rc.matrix_to_rotation_6d(rec_pose).reshape(bs, n, j * 6)
        tar_pose = rc.axis_angle_to_matrix(tar_pose.reshape(bs * n, j, 3))
        tar_pose = rc.matrix_to_rotation_6d(tar_pose).reshape(bs, n, j * 6)

        return {
            'rec_pose': rec_pose,
            'rec_trans': rec_trans,
            'tar_pose': tar_pose,
            'tar_exps': tar_exps,
            'tar_beta': tar_beta,
            'tar_trans': tar_trans,
            'rec_exps': rec_exps,
        }

    def test_demo(self, epoch):
        '''
        input audio and text, output motion
        do not calculate loss and metric
        save video
        '''
        results_save_path = self.checkpoint_path + f"/{epoch}/"
        if os.path.exists(results_save_path):
            import shutil
            shutil.rmtree(results_save_path)
        os.makedirs(results_save_path)
        start_time = time.time()
        total_length = 0
        self.model.eval()
        self.smplx.eval()
        with torch.no_grad():
            for its, batch_data in enumerate(self.test_loader):
                loaded_data = self._load_data(batch_data)
                net_out = self._g_test(loaded_data)
                tar_pose = net_out['tar_pose']
                rec_pose = net_out['rec_pose']
                tar_exps = net_out['tar_exps']
                tar_beta = net_out['tar_beta']
                rec_trans = net_out['rec_trans']
                tar_trans = net_out['tar_trans']
                rec_exps = net_out['rec_exps']
                bs, n, j = tar_pose.shape[0], tar_pose.shape[1], self.joints
                if (30 / self.args.pose_fps) != 1:
                    assert 30 % self.args.pose_fps == 0
                    n *= int(30 / self.args.pose_fps)
                    tar_pose = torch.nn.functional.interpolate(
                        tar_pose.permute(0, 2, 1),
                        scale_factor=30 / self.args.pose_fps, mode='linear'
                    ).permute(0, 2, 1)
                    rec_pose = torch.nn.functional.interpolate(
                        rec_pose.permute(0, 2, 1),
                        scale_factor=30 / self.args.pose_fps, mode='linear'
                    ).permute(0, 2, 1)

                rec_pose = rc.rotation_6d_to_matrix(rec_pose.reshape(bs * n, j, 6))
                rec_pose = rc.matrix_to_rotation_6d(rec_pose).reshape(bs, n, j * 6)
                tar_pose = rc.rotation_6d_to_matrix(tar_pose.reshape(bs * n, j, 6))
                tar_pose = rc.matrix_to_rotation_6d(tar_pose).reshape(bs, n, j * 6)

                rec_pose = rc.rotation_6d_to_matrix(rec_pose.reshape(bs * n, j, 6))
                rec_pose = rc.matrix_to_axis_angle(rec_pose).reshape(bs * n, j * 3)
                tar_pose = rc.rotation_6d_to_matrix(tar_pose.reshape(bs * n, j, 6))
                tar_pose = rc.matrix_to_axis_angle(tar_pose).reshape(bs * n, j * 3)

                tar_pose_np = tar_pose.detach().cpu().numpy()
                rec_pose_np = rec_pose.detach().cpu().numpy()
                rec_trans_np = rec_trans.detach().cpu().numpy().reshape(bs * n, 3)
                rec_exp_np = rec_exps.detach().cpu().numpy().reshape(bs * n, 100)
                tar_exp_np = tar_exps.detach().cpu().numpy().reshape(bs * n, 100)
                tar_trans_np = tar_trans.detach().cpu().numpy().reshape(bs * n, 3)
                gt_npz = np.load("./demo/examples/2_scott_0_1_1.npz", allow_pickle=True)

                results_npz_file_save_path = results_save_path + f"result_{self.time_name_expend}" + '.npz'
                np.savez(
                    results_npz_file_save_path,
                    betas=gt_npz["betas"],
                    poses=rec_pose_np,
                    expressions=rec_exp_np,
                    trans=rec_trans_np,
                    model='smplx2020',
                    gender='neutral',
                    mocap_frame_rate=30,
                )
                total_length += n
                render_vid_path = other_tools_hf.render_one_sequence_no_gt(
                    results_npz_file_save_path,
                    results_save_path,
                    self.audio_path,
                    self.args.data_path_1 + "smplx_models/",
                    use_matplotlib=False,
                    args=self.args,
                )

        result = [
            gr.Video(value=render_vid_path, visible=True),
            gr.File(value=results_npz_file_save_path, label="download motion and visualize in blender"),
        ]

        end_time = time.time() - start_time
        logger.info(
            f"total inference time: {int(end_time)} s for {int(total_length / self.args.pose_fps)} s motion"
        )
        return result


@logger.catch
def gesturelsm(audio_path, sample_stratege):
    args, cfg = config.parse_args()
    print(sample_stratege)
    if not sys.warnoptions:
        warnings.simplefilter("ignore")
    other_tools_hf.set_random_seed(args)
    other_tools_hf.print_exp_info(args)
    trainer = BaseTrainer(args, cfg, ap=audio_path)
    other_tools.load_checkpoints(trainer.model, args.test_ckpt, args.g_name)
    result = trainer.test_demo(999)
    return result


examples = [
    ["demo/examples/2_scott_0_1_1.wav"],
    ["demo/examples/2_scott_0_2_2.wav"],
    ["demo/examples/2_scott_0_3_3.wav"],
    ["demo/examples/2_scott_0_4_4.wav"],
    ["demo/examples/2_scott_0_5_5.wav"],
]

demo = gr.Interface(
    gesturelsm,
    inputs=[
        gr.Audio(),
    ],
    outputs=[
        gr.Video(format="mp4", visible=True),
        gr.File(label="download motion and visualize in blender")
    ],
    title='GestureLSM: Latent Shortcut based Co-Speech Gesture Generation with Spatial-Temporal Modeling',
    description="1. Upload your audio. <br/>\
        2. Then, sit back and wait for the rendering to happen! This may take a while (e.g. 1-4 minutes) <br/>\
        3. After, you can view the videos. <br/>\
        4. Notice that we use a fix face animation, our method only produce body motion. <br/>\
        5. Use DDPM sample strategy will generate a better result, while it will take more inference time. \
            ",
    article="Project links: [GestureLSM](https://github.com/andypinxinliu/GestureLSM). <br/>\
             Reference links: [EMAGE](https://pantomatrix.github.io/EMAGE/). ",
    examples=examples,
)

if __name__ == "__main__":
    os.environ["MASTER_ADDR"] = '127.0.0.3'
    os.environ["MASTER_PORT"] = '8678'
    demo.launch(server_name="0.0.0.0", share=True)

Writing demo2.py


In [10]:
import re

filepath = "/content/GestureLSM/dataloaders/beat_sep_single.py"
with open(filepath, "r") as f:
    content = f.read()

content = content.replace(
    "from .utils.audio_features import Wav2Vec2Model",
    "# from .utils.audio_features import Wav2Vec2Model  # removed: class unused"
)

with open(filepath, "w") as f:
    f.write(content)

print("Done! Line commented out.")

Done! Line commented out.


In [11]:
filepath = "/content/GestureLSM/dataloaders/beat_sep_single.py"
with open(filepath, "r") as f:
    content = f.read()

old = """        split_rule = pd.read_csv(args.data_path+"train_test_split.csv")
        self.selected_file = split_rule.loc[(split_rule['type'] == loader_type) & (split_rule['id'].str.split("_").str[0].astype(int).isin(self.args.training_speakers))]
        if args.additional_data and loader_type == 'train':
            split_b = split_rule.loc[(split_rule['type'] == 'additional') & (split_rule['id'].str.split("_").str[0].astype(int).isin(self.args.training_speakers))]
            #self.selected_file = split_rule.loc[(split_rule['type'] == 'additional') & (split_rule['id'].str.split("_").str[0].astype(int).isin(self.args.training_speakers))]
            self.selected_file = pd.concat([self.selected_file, split_b])
        if self.selected_file.empty:
            logger.warning(f"{loader_type} is empty for speaker {self.args.training_speakers}, use train set 0-8 instead")
            self.selected_file = split_rule.loc[(split_rule['type'] == 'train') & (split_rule['id'].str.split("_").str[0].astype(int).isin(self.args.training_speakers))]
            self.selected_file = self.selected_file.iloc[0:8]"""

new = """        # Demo mode: skip BEAT dataset CSV (not needed for single-audio inference)
        csv_path = args.data_path + "train_test_split.csv"
        if loader_type == "test" and not os.path.exists(csv_path):
            logger.warning("train_test_split.csv not found — running in demo mode with dummy selected_file")
            self.selected_file = pd.DataFrame(columns=['id', 'type'])
        else:
            split_rule = pd.read_csv(csv_path)
            self.selected_file = split_rule.loc[(split_rule['type'] == loader_type) & (split_rule['id'].str.split("_").str[0].astype(int).isin(self.args.training_speakers))]
            if args.additional_data and loader_type == 'train':
                split_b = split_rule.loc[(split_rule['type'] == 'additional') & (split_rule['id'].str.split("_").str[0].astype(int).isin(self.args.training_speakers))]
                self.selected_file = pd.concat([self.selected_file, split_b])
            if self.selected_file.empty:
                logger.warning(f"{loader_type} is empty for speaker {self.args.training_speakers}, use train set 0-8 instead")
                self.selected_file = split_rule.loc[(split_rule['type'] == 'train') & (split_rule['id'].str.split("_").str[0].astype(int).isin(self.args.training_speakers))]
                self.selected_file = self.selected_file.iloc[0:8]"""

assert old in content, "❌ Pattern not found — check indentation or line endings"
content = content.replace(old, new)

with open(filepath, "w") as f:
    f.write(content)

print("✅ Patch applied successfully")

✅ Patch applied successfully


In [12]:
!grep -E "word_rep|word_cache|t_pre_encoder" /content/GestureLSM/configs/shortcut_rvqvae_128_hf.yaml

word_rep: textgrid
t_pre_encoder: fasttext


In [13]:
import os
# Check if vocab.pkl exists anywhere in the repo
for root, dirs, files in os.walk("/content/GestureLSM"):
    for f in files:
        if "vocab" in f:
            print(os.path.join(root, f))

# Also check what's in data_path_1
print(os.path.exists("./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/weights/"))

/content/GestureLSM/dataloaders/build_vocab.py
/content/GestureLSM/models/utils/build_vocab.py
/content/GestureLSM/weights/vocab.pkl
False


In [14]:
import os

# Check what's in the repo's weights folder
weights_dir = "/content/GestureLSM/weights"
print("=== GestureLSM/weights/ ===")
for root, dirs, files in os.walk(weights_dir):
    for f in files:
        fpath = os.path.join(root, f)
        size = os.path.getsize(fpath)
        print(f"  {fpath}  ({size/1024:.1f} KB)")

# Check what the code expects at data_path/weights/
expected_dir = "./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/weights/"
print(f"\n=== Expected path exists: {os.path.exists(expected_dir)} ===")
if os.path.exists(expected_dir):
    for f in os.listdir(expected_dir):
        print(f"  {f}")

# Also check data_path_1
print("\n=== GestureLSM/datasets/ (top level) ===")
datasets_dir = "/content/GestureLSM/datasets"
if os.path.exists(datasets_dir):
    for root, dirs, files in os.walk(datasets_dir):
        for f in files:
            print(os.path.join(root, f))
else:
    print("  Not found")

=== GestureLSM/weights/ ===
  /content/GestureLSM/weights/mean_vel_smplxflame_30.npy  (0.3 KB)
  /content/GestureLSM/weights/AESKConv_240_100.bin  (17147.1 KB)
  /content/GestureLSM/weights/vocab.pkl  (13497.4 KB)

=== Expected path exists: False ===

=== GestureLSM/datasets/ (top level) ===
/content/GestureLSM/datasets/hub/pretrained_vq/face_vertex_1layer_790.bin
/content/GestureLSM/datasets/hub/smplx_models/smplx/SMPLX_NEUTRAL_2020.npz


In [15]:
import os

# The path the code expects
expected = "./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0"
os.makedirs(expected, exist_ok=True)

# Symlink weights/ folder
weights_link = os.path.join(expected, "weights")
if not os.path.exists(weights_link):
    os.symlink("/content/GestureLSM/weights", weights_link)
    print(f"✅ Linked weights/")
else:
    print(f"⚠️  weights/ link already exists")

# Verify
print("\n=== Verifying symlinks ===")
for f in os.listdir(weights_link):
    print(f"  {weights_link}/{f}")

✅ Linked weights/

=== Verifying symlinks ===
  ./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/weights/mean_vel_smplxflame_30.npy
  ./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/weights/AESKConv_240_100.bin
  ./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/weights/vocab.pkl


In [16]:
import os

# Check what data_path_1 is set to in the config
!grep "data_path_1\|data_path\b" /content/GestureLSM/configs/shortcut_rvqvae_128_hf.yaml

# Pre-emptively symlink smplx_models if needed
# First check where code expects smplx models
!grep -r "smplx_models" /content/GestureLSM/demo2.py | head -5

data_path: ./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/
data_path_1: ./datasets/hub/
            self.args.data_path_1 + "smplx_models/",
                    self.args.data_path_1 + "smplx_models/",


In [17]:
!grep -E "data_path|data_path_1|args\." /content/GestureLSM/configs/shortcut_rvqvae_128_hf.yaml | grep "path"

data_path: ./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/
data_path_1: ./datasets/hub/


In [18]:
import os

print("=== Setting up all required symlinks ===\n")

symlinks = [
    # (target that exists, link path the code expects)

    # data_path weights (already done, will skip gracefully)
    (
        "/content/GestureLSM/weights",
        "./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/weights"
    ),

    # data_path_1 smplx_models
    (
        "/content/GestureLSM/datasets/hub/smplx_models",
        "./datasets/hub/smplx_models"
    ),

    # data_path_1 hub/pretrained_vq (used by some model loaders)
    (
        "/content/GestureLSM/datasets/hub/pretrained_vq",
        "./datasets/hub/pretrained_vq"
    ),
]

for target, link in symlinks:
    # Make sure parent dir exists
    os.makedirs(os.path.dirname(os.path.abspath(link)), exist_ok=True)

    if os.path.exists(link) or os.path.islink(link):
        print(f"⚠️  Already exists, skipping: {link}")
    elif not os.path.exists(target):
        print(f"❌ Target not found, can't link: {target}")
    else:
        os.symlink(target, link)
        print(f"✅ Linked: {link} → {target}")

print("\n=== Verification ===")
for target, link in symlinks:
    exists = os.path.exists(link)
    print(f"  {'✅' if exists else '❌'} {link}")

=== Setting up all required symlinks ===

⚠️  Already exists, skipping: ./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/weights
⚠️  Already exists, skipping: ./datasets/hub/smplx_models
⚠️  Already exists, skipping: ./datasets/hub/pretrained_vq

=== Verification ===
  ✅ ./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/weights
  ✅ ./datasets/hub/smplx_models
  ✅ ./datasets/hub/pretrained_vq


In [19]:
import os

# Scan demo2.py and beat_sep_single.py for all path references
# so we can fix everything remaining in one shot
paths_to_check = [
    "./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/",
    "./datasets/hub/",
    "./datasets/hub/smplx_models/",
    "./datasets/hub/pretrained_vq/",
    "./demo/examples/2_scott_0_1_1.npz",
    "./demo/examples/2_scott_0_1_1.wav",
]

print("=== Path existence check ===")
for p in paths_to_check:
    exists = os.path.exists(p)
    print(f"  {'✅' if exists else '❌'} {p}")

# Also check demo/ folder
print("\n=== ./demo/ contents ===")
if os.path.exists("./demo"):
    for root, dirs, files in os.walk("./demo"):
        for f in files:
            print(f"  {os.path.join(root, f)}")
else:
    print("  ❌ ./demo/ not found")

# Check outputs dir
print("\n=== ./outputs/ exists:", os.path.exists("./outputs/"))

=== Path existence check ===
  ✅ ./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/
  ✅ ./datasets/hub/
  ✅ ./datasets/hub/smplx_models/
  ✅ ./datasets/hub/pretrained_vq/
  ✅ ./demo/examples/2_scott_0_1_1.npz
  ✅ ./demo/examples/2_scott_0_1_1.wav

=== ./demo/ contents ===
  ./demo/install_mfa.sh
  ./demo/examples/2_scott_0_3_3.wav
  ./demo/examples/2_scott_0_2_2.wav
  ./demo/examples/2_scott_0_1_1.wav
  ./demo/examples/2_scott_0_5_5.wav
  ./demo/examples/2_scott_0_4_4.wav
  ./demo/examples/2_scott_0_1_1.npz

=== ./outputs/ exists: False


In [20]:
import os

print("=== /content/GestureLSM/models/ contents ===")
for root, dirs, files in os.walk("/content/GestureLSM/models"):
    # skip __pycache__
    if "__pycache__" in root:
        continue
    for f in files:
        print(os.path.join(root, f))

=== /content/GestureLSM/models/ contents ===
/content/GestureLSM/models/MeanFlow.py
/content/GestureLSM/models/motion_representation.py
/content/GestureLSM/models/motion_encoder.py
/content/GestureLSM/models/Diffusion.py
/content/GestureLSM/models/quantizer.py
/content/GestureLSM/models/LSM.py
/content/GestureLSM/models/config.py
/content/GestureLSM/models/denoiser.py
/content/GestureLSM/models/utils/wav2vec.py
/content/GestureLSM/models/utils/fk.py
/content/GestureLSM/models/utils/layer.py
/content/GestureLSM/models/utils/utils.py
/content/GestureLSM/models/utils/build_vocab.py
/content/GestureLSM/models/utils/__init__.py
/content/GestureLSM/models/utils/rotation_conversions.py
/content/GestureLSM/models/utils/audio_utils.py
/content/GestureLSM/models/utils/rotations.py
/content/GestureLSM/models/utils/skeleton.py
/content/GestureLSM/models/wavlm/modules_wavlm.py
/content/GestureLSM/models/wavlm/WavLM.py
/content/GestureLSM/models/layers/layer.py
/content/GestureLSM/models/layers/util

In [21]:
!grep -A5 "modality_encoder" /content/GestureLSM/configs/shortcut_rvqvae_128_hf.yaml

In [22]:
!sed -n '110,135p' /content/GestureLSM/models/LSM.py

    return t

def reshape_coefs(t):
    return t.reshape((t.shape[0], 1, 1, 1))

class GestureLSM(torch.nn.Module):
    def __init__(self, cfg) -> None:
        super().__init__()
        self.cfg = cfg

        # Initialize model components
        self.modality_encoder = instantiate_from_config(cfg.model.modality_encoder)
        self.denoiser = instantiate_from_config(cfg.model.denoiser)

        # Model hyperparameters
        self.do_classifier_free_guidance = cfg.model.do_classifier_free_guidance
        self.guidance_scale = cfg.model.guidance_scale
        self.num_inference_steps = cfg.model.n_steps

        # Loss functions
        self.smooth_l1_loss = torch.nn.SmoothL1Loss(reduction='none')
        
        self.num_joints = self.denoiser.joint_num
        
        self.seq_len = self.denoiser.seq_len
        self.input_dim = self.denoiser.input_dim


In [23]:
import re

config_path = "/content/GestureLSM/configs/shortcut_rvqvae_128_hf.yaml"
with open(config_path, "r") as f:
    content = f.read()

# Preview what we'd change
import re
matches = re.findall(r'models\.modality_encoder', content)
print(f"Found {len(matches)} occurrence(s) of 'models.modality_encoder'")
print("Would replace with: 'models.layers.modality_encoder'")

Found 0 occurrence(s) of 'models.modality_encoder'
Would replace with: 'models.layers.modality_encoder'


In [24]:
# Search everywhere for where modality_encoder target is defined
!grep -r "modality_encoder" /content/GestureLSM/configs/
!echo "---"
# Check all yaml/config files
!find /content/GestureLSM -name "*.yaml" -o -name "*.yml" | xargs grep -l "modality_encoder" 2>/dev/null
!echo "---"
# Maybe it's hardcoded in LSM.py itself
!grep -n "modality_encoder" /content/GestureLSM/models/LSM.py

/content/GestureLSM/configs/model_config.yaml:  modality_encoder:
/content/GestureLSM/configs/model_config.yaml:    target: models.modality_encoder.ModalityEncoder
/content/GestureLSM/configs/sc_model_config.yaml:  modality_encoder:
/content/GestureLSM/configs/sc_model_config.yaml:    target: models.modality_encoder.ModalityEncoder
/content/GestureLSM/configs/sc_reflow_model_config.yaml:  modality_encoder:
/content/GestureLSM/configs/sc_reflow_model_config.yaml:    target: models.modality_encoder.ModalityEncoder
/content/GestureLSM/configs/sc_model_holistic_config.yaml:  modality_encoder:
/content/GestureLSM/configs/sc_model_holistic_config.yaml:    target: models.modality_encoder.ModalityEncoder
---
/content/GestureLSM/configs_new/diffusion_rvqvae_128.yaml
/content/GestureLSM/configs_new/shortcut_rvqvae_128.yaml
/content/GestureLSM/configs_new/meanflow_rvqvae_128.yaml
/content/GestureLSM/configs/model_config.yaml
/content/GestureLSM/configs/sc_model_config.yaml
/content/GestureLSM/con

In [25]:
import os

config_files = [
    "/content/GestureLSM/configs/sc_reflow_model_config.yaml",
    "/content/GestureLSM/configs/model_config.yaml",
    "/content/GestureLSM/configs/sc_model_config.yaml",
    "/content/GestureLSM/configs/sc_model_holistic_config.yaml",
]

old = "target: models.modality_encoder.ModalityEncoder"
new = "target: models.layers.modality_encoder.ModalityEncoder"

for path in config_files:
    with open(path, "r") as f:
        content = f.read()
    if old in content:
        content = content.replace(old, new)
        with open(path, "w") as f:
            f.write(content)
        print(f"✅ Fixed: {path}")
    else:
        print(f"⚠️  Pattern not found in: {path}")

✅ Fixed: /content/GestureLSM/configs/sc_reflow_model_config.yaml
✅ Fixed: /content/GestureLSM/configs/model_config.yaml
✅ Fixed: /content/GestureLSM/configs/sc_model_config.yaml
✅ Fixed: /content/GestureLSM/configs/sc_model_holistic_config.yaml


In [26]:
# import shutil, sys

# INPUT = "/content/GestureLSM/demo2.py"   # ← change if your path is different

# # ── read ─────────────────────────────────────────────────────────────────────
# with open(INPUT, "r") as f:
#     src = f.read()

# original = src
# errors = []

# # ── Fix 1 : inject audio_onset ───────────────────────────────────────────────
# OLD = "            cond_['y']['audio'] = in_audio_tmp"

# NEW = """\
#             cond_['y']['audio'] = in_audio_tmp
#             # ── Fix 1: build audio_onset so LSM.py line 270 doesn't KeyError ─
#             _audio_np = in_audio_tmp.squeeze().cpu().numpy()
#             if _audio_np.ndim == 2:
#                 _audio_np = _audio_np.mean(axis=0)          # stereo → mono
#             import librosa as _librosa, numpy as _np
#             _onset_env = _librosa.onset.onset_strength(
#                 y=_audio_np, sr=16000, hop_length=160
#             )
#             _target_len = in_audio_tmp.shape[-1]
#             _onset_env = _np.interp(
#                 _np.linspace(0, len(_onset_env) - 1, _target_len),
#                 _np.arange(len(_onset_env)),
#                 _onset_env,
#             ).astype(_np.float32)
#             cond_['y']['audio_onset'] = (
#                 torch.from_numpy(_onset_env).unsqueeze(0).unsqueeze(0).to(self.rank)
#             )
#             # ─────────────────────────────────────────────────────────────────"""

# if "audio_onset" in src:
#     print("⚠️  Fix 1 SKIPPED — audio_onset already in file")
# elif OLD not in src:
#     errors.append("Fix 1: could not find cond_['y']['audio'] line — check indentation or file path")
# else:
#     src = src.replace(OLD, NEW, 1)
#     print("✅  Fix 1 applied — audio_onset injected")

# # ── Fix 2 : guard Gradio return value ────────────────────────────────────────
# OLD2 = "    result = trainer.test_demo(999)\n    return result"

# NEW2 = """\
#     result = trainer.test_demo(999)
#     # ── Fix 2: guarantee Gradio always receives exactly 2 outputs ────────────
#     if result is None:
#         return [None, None]
#     # ─────────────────────────────────────────────────────────────────────────
#     return result"""

# if "[None, None]" in src:
#     print("⚠️  Fix 2 SKIPPED — guard already in file")
# elif OLD2 not in src:
#     errors.append("Fix 2: could not find 'result = trainer.test_demo(999)' line")
# else:
#     src = src.replace(OLD2, NEW2, 1)
#     print("✅  Fix 2 applied — Gradio return guard added")

# # ── write / report ────────────────────────────────────────────────────────────
# if errors:
#     print("\n❌ Errors — no changes written:")
#     for e in errors:
#         print("   •", e)
#     sys.exit(1)

# if src == original:
#     print("\nℹ️  File already up to date — nothing to write")
# else:
#     shutil.copy(INPUT, INPUT + ".bak")
#     print(f"💾  Backup saved → {INPUT}.bak")
#     with open(INPUT, "w") as f:
#         f.write(src)
#     print(f"✅  Patched file saved → {INPUT}")

# print("\n🚀  Done! You can now run new_demo.py")

In [27]:
# import shutil, sys

# INPUT = "/content/GestureLSM/demo2.py"   # ← change if your path is different

# # ── read ─────────────────────────────────────────────────────────────────────
# with open(INPUT, "r") as f:
#     src = f.read()

# original = src
# errors = []

# # ─────────────────────────────────────────────────────────────────────────────
# # Fix 1 (v1 cleanup): remove the wrong librosa onset block if already applied
# # ─────────────────────────────────────────────────────────────────────────────
# WRONG_BLOCK_MARKER = "# ── Fix 1: build audio_onset so LSM.py line 270 doesn't KeyError ─"

# if WRONG_BLOCK_MARKER in src:
#     # Remove the entire injected block — everything from the marker line up to
#     # and including the closing dashes comment, then the cond_ assignment line
#     import re
#     src = re.sub(
#         r"            # ── Fix 1: build audio_onset.*?# ─{60,}\n",
#         "",
#         src,
#         flags=re.DOTALL,
#     )
#     print("🧹  Removed old (wrong) Fix 1 librosa onset block")

# # ─────────────────────────────────────────────────────────────────────────────
# # Fix 1 (correct): pass raw audio as audio_onset — that is what LSM.py needs.
# # LSM.py uses condition_dict['y']['audio_onset'] as the main audio fed into
# # WavEncoder. The key name is just what the model calls it internally; it
# # expects the same raw waveform tensor that was previously stored under 'audio'.
# # ─────────────────────────────────────────────────────────────────────────────
# OLD_AUDIO = "            cond_['y']['audio'] = in_audio_tmp"
# NEW_AUDIO  = """\
#             cond_['y']['audio'] = in_audio_tmp
#             # Fix 1: LSM.py reads audio from 'audio_onset', not 'audio'
#             cond_['y']['audio_onset'] = in_audio_tmp"""

# if "cond_['y']['audio_onset'] = in_audio_tmp" in src:
#     print("⚠️  Fix 1 SKIPPED — audio_onset already correctly set")
# elif OLD_AUDIO not in src:
#     errors.append("Fix 1: could not find cond_['y']['audio'] = in_audio_tmp  — check file path / indentation")
# else:
#     src = src.replace(OLD_AUDIO, NEW_AUDIO, 1)
#     print("✅  Fix 1 applied — audio_onset now points to raw audio (in_audio_tmp)")

# # ─────────────────────────────────────────────────────────────────────────────
# # Fix 2: guard Gradio return so it always gets exactly [video, file]
# # ─────────────────────────────────────────────────────────────────────────────
# OLD_RETURN = "    result = trainer.test_demo(999)\n    return result"
# NEW_RETURN = """\
#     result = trainer.test_demo(999)
#     # Fix 2: guarantee Gradio always receives exactly 2 outputs
#     if result is None:
#         return [None, None]
#     return result"""

# if "[None, None]" in src:
#     print("⚠️  Fix 2 SKIPPED — Gradio guard already present")
# elif OLD_RETURN not in src:
#     errors.append("Fix 2: could not find 'result = trainer.test_demo(999)' block")
# else:
#     src = src.replace(OLD_RETURN, NEW_RETURN, 1)
#     print("✅  Fix 2 applied — Gradio return guard added")

# # ─────────────────────────────────────────────────────────────────────────────
# # Write / report
# # ─────────────────────────────────────────────────────────────────────────────
# if errors:
#     print("\n❌  Errors — no changes written:")
#     for e in errors:
#         print("   •", e)
#     sys.exit(1)

# if src == original:
#     print("\nℹ️  File already up to date — nothing written")
# else:
#     shutil.copy(INPUT, INPUT + ".bak")
#     print(f"💾  Backup saved  → {INPUT}.bak")
#     with open(INPUT, "w") as f:
#         f.write(src)
#     print(f"✅  Patched file  → {INPUT}")

# print("\n🚀  Done! You can now run new_demo.py")

In [28]:
# See what keys are being put into condition_dict in demo2.py
!grep -n "audio_onset\|audio\|condition_dict\|cond_" /content/GestureLSM/demo2.py | head -40

# See how LSM.py uses audio_onset
!grep -n "audio_onset\|audio" /content/GestureLSM/models/LSM.py | head -40

128:def _run_whisperx_alignment(audio_path, tmp_dir, textgrid_path):
139:    audio_array = whisperx.load_audio(audio_path)
140:    result = whisperx_asr_model.transcribe(audio_array, batch_size=8)
158:        audio_array,
165:    duration = len(audio_array) / 16000.0   # whisperx always uses 16 kHz
190:        self.audio_path = tmp_dir + "/tmp.wav"
191:        sf.write(self.audio_path, ap[1], ap[0])
193:        audio, ssr = librosa.load(self.audio_path, sr=args.audio_sr)
199:            _run_whisperx_alignment(self.audio_path, tmp_dir, self.textgrid_path)
202:        ap = (ssr, audio)
207:        args.audio_file_path = self.audio_path
380:        in_audio = dict_data["audio"].to(self.rank)
424:            "in_audio": in_audio,
447:        in_audio = loaded_data["in_audio"]
502:            in_audio_tmp = in_audio[
530:            cond_ = {'y': {}}
531:            cond_['y']['audio'] = in_audio_tmp
532:            cond_['y']['word'] = in_word_tmp
533:            cond_['y']['id'] = in_id_

In [29]:
# See what keys are actually in the batch coming from the dataloader
!sed -n '530,555p' /content/GestureLSM/demo2.py

            cond_ = {'y': {}}
            cond_['y']['audio'] = in_audio_tmp
            cond_['y']['word'] = in_word_tmp
            cond_['y']['id'] = in_id_tmp
            cond_['y']['seed'] = in_seed_tmp
            cond_['y']['mask'] = (
                torch.zeros([self.args.batch_size, 1, 1, self.args.pose_length]) < 1
            ).cuda()
            cond_['y']['style_feature'] = torch.zeros([bs, 512]).cuda()

            shape_ = (bs, 3 * 128, 1, 32)
            sample = self.model(cond_)['latents']
            sample = sample.squeeze().permute(1, 0).unsqueeze(0)

            last_sample = sample.clone()

            rec_latent_upper = sample[..., :128]
            rec_latent_hands = sample[..., 128:2 * 128]
            rec_latent_lower = sample[..., 2 * 128:]

            if i == 0:
                rec_all_upper.append(rec_latent_upper)
                rec_all_hands.append(rec_latent_hands)
                rec_all_lower.append(rec_latent_lower)
            else:
             

In [30]:
filepath = "/content/GestureLSM/demo2.py"
with open(filepath, "r") as f:
    content = f.read()

old = "cond_['y']['audio'] = in_audio_tmp"
new = "cond_['y']['audio_onset'] = in_audio_tmp"

assert old in content, "❌ Pattern not found"
content = content.replace(old, new)

with open(filepath, "w") as f:
    f.write(content)

print("✅ Fixed: cond_['y']['audio'] → cond_['y']['audio_onset']")

✅ Fixed: cond_['y']['audio'] → cond_['y']['audio_onset']


In [31]:
# See the full modality_encoder forward pass
!cat /content/GestureLSM/models/layers/modality_encoder.py

# Check the relevant config dimensions
!grep -E "audio_f|word_f|audio_dims|word_dims|hidden|feat_dim|encoder_dim|audio_rep" /content/GestureLSM/configs/shortcut_rvqvae_128_hf.yaml

import pdb

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from models.layers.layer import BasicBlock
from einops import rearrange
import pickle
import math
from models.wavlm.WavLM import WavLM, WavLMConfig


class ExactLengthAdjuster(nn.Module):
    """
    Layer that ensures the output has exactly the target length along the time dimension.
    It either adds or removes frames as needed.
    """
    def __init__(self, target_length=196):
        super(ExactLengthAdjuster, self).__init__()
        self.target_length = target_length
    
    def forward(self, x):
        # x is expected to be [batch, channels, time]
        current_length = x.shape[2]
        
        if current_length == self.target_length:
            return x
        elif current_length < self.target_length:
            # Need to add frames
            frames_to_add = self.target_length - current_length
            
            # Duplicate the last frame as many times as needed

In [32]:
# Check the actual config being used
!cat /content/GestureLSM/configs/shortcut_rvqvae_128_hf.yaml | grep -E "audio_dim|audio_f|audio_in|latent_dim|word_dim|word_f"

# Also check what params are passed to ModalityEncoder in the model config
!grep -A 20 "modality_encoder" /content/GestureLSM/configs/sc_model_config.yaml

# And check which model config gets merged in
!grep -E "model_config\|sc_model\|merge\|include" /content/GestureLSM/configs/shortcut_rvqvae_128_hf.yaml | head -20

audio_fps: 16000
audio_f: 256
word_dims: 300
  modality_encoder:
    target: models.layers.modality_encoder.ModalityEncoder
    params:
      data_path: ./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/
      t_fix_pre: False
      audio_dim: 256
      audio_in: 2
      raw_audio: False
      latent_dim: 256
      audio_fps: 30
      use_exp: ${model.use_exp}
  


In [33]:
# See how in_word_tmp is sliced in _g_test
!grep -n "in_word\|pose_length\|pre_frames\|stride\|seq_len\|target_length" /content/GestureLSM/demo2.py | head -30

# Check config values
!grep -E "pose_length|pre_frames|stride|target_length|seq_len" /content/GestureLSM/configs/shortcut_rvqvae_128_hf.yaml

351:            args.down_t, args.stride_t, args.width, args.depth,
385:        in_word = dict_data["word"].to(self.rank)
426:            "in_word": in_word,
446:        in_word = loaded_data["in_word"]
456:            in_word = in_word[:, :-remain]
490:        roundt = (n - self.args.pre_frames * vqvae_squeeze_scale) // (
491:            self.args.pose_length - self.args.pre_frames * vqvae_squeeze_scale
493:        remain = (n - self.args.pre_frames * vqvae_squeeze_scale) % (
494:            self.args.pose_length - self.args.pre_frames * vqvae_squeeze_scale
496:        round_l = self.args.pose_length - self.args.pre_frames * vqvae_squeeze_scale
499:            in_word_tmp = in_word[
500:                :, i * round_l:(i + 1) * round_l + self.args.pre_frames * vqvae_squeeze_scale
505:                (i + 1) * (16000 // 30 * round_l) + 16000 // 30 * self.args.pre_frames * vqvae_squeeze_scale
508:                :, i * round_l:(i + 1) * round_l + self.args.pre_frames
513:                

In [34]:
config_path = "/content/GestureLSM/configs/sc_model_config.yaml"
with open(config_path, "r") as f:
    content = f.read()

print(content[content.find("modality_encoder"):content.find("modality_encoder")+500])

modality_encoder:
    target: models.layers.modality_encoder.ModalityEncoder
    params:
      data_path: ./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/
      t_fix_pre: False
      audio_dim: 256
      audio_in: 2
      raw_audio: False
      latent_dim: 256
      audio_fps: 30
      use_exp: ${model.use_exp}
  


In [35]:
import re

files_to_check = [
    "/content/GestureLSM/configs/sc_model_config.yaml",
    "/content/GestureLSM/configs/sc_reflow_model_config.yaml",
    "/content/GestureLSM/configs/model_config.yaml",
    "/content/GestureLSM/configs/sc_model_holistic_config.yaml",
]

for path in files_to_check:
    with open(path, "r") as f:
        content = f.read()
    if "modality_encoder" in content:
        snippet = content[content.find("modality_encoder"):content.find("modality_encoder")+400]
        print(f"\n=== {path} ===")
        print(snippet)


=== /content/GestureLSM/configs/sc_model_config.yaml ===
modality_encoder:
    target: models.layers.modality_encoder.ModalityEncoder
    params:
      data_path: ./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/
      t_fix_pre: False
      audio_dim: 256
      audio_in: 2
      raw_audio: False
      latent_dim: 256
      audio_fps: 30
      use_exp: ${model.use_exp}
  

=== /content/GestureLSM/configs/sc_reflow_model_config.yaml ===
modality_encoder:
    target: models.layers.modality_encoder.ModalityEncoder
    params:
      data_path: ./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/
      t_fix_pre: False
      audio_dim: 256
      audio_in: 2
      raw_audio: False
      latent_dim: 256
      audio_fps: 30
      use_exp: ${model.use_exp}
  

=== /content/GestureLSM/configs/model_config.yaml ===
modality_encoder:
    target: models.layers.modality_encoder.ModalityEncoder
    params:
      data_path: ./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/
      t_fix_pre: F

In [36]:
files = [
    "/content/GestureLSM/configs/sc_model_config.yaml",
    "/content/GestureLSM/configs/sc_reflow_model_config.yaml",
    "/content/GestureLSM/configs/model_config.yaml",
    "/content/GestureLSM/configs/sc_model_holistic_config.yaml",
]

old = "      audio_fps: 30"
new = "      audio_fps: 30\n      target_length: 128"

for path in files:
    with open(path, "r") as f:
        content = f.read()
    if "target_length" in content:
        print(f"⚠️  Already has target_length, skipping: {path}")
    elif old in content:
        content = content.replace(old, new, 1)  # only first occurrence
        with open(path, "w") as f:
            f.write(content)
        print(f"✅ Fixed: {path}")
    else:
        print(f"❌ Pattern not found in: {path}")

✅ Fixed: /content/GestureLSM/configs/sc_model_config.yaml
✅ Fixed: /content/GestureLSM/configs/sc_reflow_model_config.yaml
✅ Fixed: /content/GestureLSM/configs/model_config.yaml
✅ Fixed: /content/GestureLSM/configs/sc_model_holistic_config.yaml


In [37]:
!sed -n '180,260p' /content/GestureLSM/models/LSM.py

        
        # Properly expand timesteps to match doubled batch size
        batch_size = x.shape[0]
        timesteps_doubled = timesteps.expand(batch_size * 2)
        
        if cond_time is not None:
            cond_time_doubled = cond_time.expand(batch_size * 2)
        else:
            cond_time_doubled = None
        
        # Create conditional and unconditional audio features
        batch_size = at_feat.shape[0]
        null_cond_embed = self.denoiser.null_cond_embed.to(at_feat.dtype)
        at_feat_uncond = null_cond_embed.unsqueeze(0).expand(batch_size, -1, -1)
        at_feat_combined = torch.cat([at_feat, at_feat_uncond], dim=0)
        
        # Run both conditional and unconditional predictions
        output = self.denoiser(
            x=x_doubled,
            timesteps=timesteps_doubled,
            seed=seed_doubled,
            at_feat=at_feat_combined,
            cond_time=cond_time_doubled,
        )
        
        # Split predictions and apply guida

In [38]:
!sed -n '260,320p' /content/GestureLSM/models/LSM.py

        """Forward pass for inference.
        
        Args:
            condition_dict: Dictionary containing input conditions including audio, word tokens,
                          and other features
        
        Returns:
            Dictionary containing generated latents
        """
        # Extract input features
        audio = condition_dict['y']['audio_onset']
        word_tokens = condition_dict['y']['word']
        ids = condition_dict['y']['id']
        seed_vectors = condition_dict['y']['seed']
        style_features = condition_dict['y']['style_feature']
        if 'wavlm' in condition_dict['y']:
            wavlm_features = condition_dict['y']['wavlm']
        else:
            wavlm_features = None
        
        return_dict = {}
        return_dict['seed'] = seed_vectors
        
        # Encode input modalities
        audio_features = self.modality_encoder(audio, word_tokens, wavlm_features)
        return_dict['at_feat'] = audio_features

        # Initiali

In [39]:
# Check null_cond_embed shape in the checkpoint
import torch
import glob

ckpts = glob.glob("/content/GestureLSM/**/*.bin", recursive=True) + \
        glob.glob("/content/GestureLSM/**/*.pth", recursive=True) + \
        glob.glob("/content/GestureLSM/**/*.ckpt", recursive=True)

print("Checkpoints found:")
for c in ckpts:
    print(c)

# Check null_cond_embed in denoiser
!grep -n "null_cond_embed\|seq_len\|target_length" /content/GestureLSM/models/denoiser.py | head -30

Checkpoints found:
/content/GestureLSM/ckpt/new_540_shortcut.bin
/content/GestureLSM/ckpt/shortcut_reflow.bin
/content/GestureLSM/ckpt/all_speakers_denoiser.bin
/content/GestureLSM/ckpt/new_540_diffusion.bin
/content/GestureLSM/datasets/hub/pretrained_vq/face_vertex_1layer_790.bin
/content/GestureLSM/datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/weights/AESKConv_240_100.bin
/content/GestureLSM/weights/AESKConv_240_100.bin
/content/GestureLSM/ckpt/net_300000_lower.pth
/content/GestureLSM/ckpt/net_300000_upper.pth
/content/GestureLSM/ckpt/hand_all_speakers.pth
/content/GestureLSM/ckpt/upper_all_speakers.pth
/content/GestureLSM/ckpt/lower_all_speakers.pth
/content/GestureLSM/ckpt/net_300000_hands.pth
/content/GestureLSM/ckpt/net_300000_face.pth
25:        seq_len=32,
53:        self.seq_len = seq_len
71:        self.null_cond_embed = nn.Parameter(torch.zeros(self.seq_len, self.latent_dim*self.joint_num), requires_grad=True)
111:        embed_style_2 = (emb_seed + emb_t).unsqueeze(1).

In [40]:
!grep -n "null_cond_embed\|seq_len" /content/GestureLSM/configs/shortcut_rvqvae_128_hf.yaml
!grep -n "null_cond_embed\|seq_len" /content/GestureLSM/configs/sc_model_config.yaml

In [41]:
import torch

ckpt = torch.load("/content/GestureLSM/ckpt/new_540_shortcut.bin", map_location="cpu")

# Find null_cond_embed and any length-related keys
for k, v in ckpt.items():
    if "null_cond" in k or "length" in k or "seq_len" in k:
        try:
            print(f"{k}: {v.shape}")
        except:
            print(f"{k}: {type(v)}")

# Also check at_feat related keys
keys = list(ckpt.keys())
print("Total keys:", len(keys))
print("First 10:", keys[:10])

Total keys: 1
First 10: ['model_state']


In [42]:
!grep "target_length" /content/GestureLSM/configs/sc_model_config.yaml

      target_length: 128


In [43]:
import torch

ckpt = torch.load("/content/GestureLSM/ckpt/new_540_shortcut.bin", map_location="cpu")
state = ckpt["model_state"]

for k, v in state.items():
    if "null_cond" in k or "seq_len" in k or "length_adjuster" in k or "target_length" in k:
        try:
            print(f"{k}: {v.shape}")
        except:
            print(f"{k}: {type(v)}")

print("---")

# Also show WavEncoder related keys
for k, v in state.items():
    if "WavEncoder" in k or "wav_encoder" in k or "length_adj" in k:
        try:
            print(f"{k}: {v.shape}")
        except:
            print(f"{k}: {type(v)}")

module.denoiser.null_cond_embed: torch.Size([32, 768])
---
module.modality_encoder.WavEncoder.feat_extractor.0.conv1.weight: torch.Size([64, 2, 15])
module.modality_encoder.WavEncoder.feat_extractor.0.conv1.bias: torch.Size([64])
module.modality_encoder.WavEncoder.feat_extractor.0.bn1.weight: torch.Size([64])
module.modality_encoder.WavEncoder.feat_extractor.0.bn1.bias: torch.Size([64])
module.modality_encoder.WavEncoder.feat_extractor.0.bn1.running_mean: torch.Size([64])
module.modality_encoder.WavEncoder.feat_extractor.0.bn1.running_var: torch.Size([64])
module.modality_encoder.WavEncoder.feat_extractor.0.bn1.num_batches_tracked: torch.Size([])
module.modality_encoder.WavEncoder.feat_extractor.0.conv2.weight: torch.Size([64, 64, 15])
module.modality_encoder.WavEncoder.feat_extractor.0.conv2.bias: torch.Size([64])
module.modality_encoder.WavEncoder.feat_extractor.0.bn2.weight: torch.Size([64])
module.modality_encoder.WavEncoder.feat_extractor.0.bn2.bias: torch.Size([64])
module.modali

In [44]:
files = [
    "/content/GestureLSM/configs/sc_model_config.yaml",
    "/content/GestureLSM/configs/sc_reflow_model_config.yaml",
    "/content/GestureLSM/configs/model_config.yaml",
    "/content/GestureLSM/configs/sc_model_holistic_config.yaml",
]

for path in files:
    with open(path, "r") as f:
        content = f.read()
    if "audio_dim: 256" in content:
        content = content.replace("audio_dim: 256", "audio_dim: 384")
        with open(path, "w") as f:
            f.write(content)
        print(f"✅ Fixed audio_dim: {path}")
    else:
        print(f"⚠️  audio_dim: 256 not found in: {path}")

# Verify
!grep "audio_dim\|target_length" /content/GestureLSM/configs/sc_model_config.yaml

✅ Fixed audio_dim: /content/GestureLSM/configs/sc_model_config.yaml
✅ Fixed audio_dim: /content/GestureLSM/configs/sc_reflow_model_config.yaml
✅ Fixed audio_dim: /content/GestureLSM/configs/model_config.yaml
✅ Fixed audio_dim: /content/GestureLSM/configs/sc_model_holistic_config.yaml
      audio_dim: 384
      target_length: 128


In [45]:
# Check which model config gets included by the main config
!cat /content/GestureLSM/configs/shortcut_rvqvae_128_hf.yaml

# Also verify the fix actually landed
!grep "audio_dim\|target_length" /content/GestureLSM/configs/sc_model_config.yaml
!grep "audio_dim\|target_length" /content/GestureLSM/configs/sc_reflow_model_config.yaml

is_train: True
ddp: False
stat: ts
root_path: ./
out_path: ./outputs/audio2pose/
project: s2g
e_path:  weights/AESKConv_240_100.bin
eval_model: motion_representation
e_name: VAESKConv
data_path: ./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/
test_ckpt: ./ckpt/new_540_shortcut.bin
data_path_1: ./datasets/hub/
pose_norm: True
cfg: configs/sc_model_config.yaml


mean_pose_path: ./mean_std/beatx_2_330_mean.npy
std_pose_path: ./mean_std/beatx_2_330_std.npy

mean_trans_path: ./mean_std/beatx_2_trans_mean.npy
std_trans_path: ./mean_std/beatx_2_trans_std.npy


vqvae_upper_path: ./ckpt/net_300000_upper.pth
vqvae_hands_path: ./ckpt/net_300000_hands.pth
vqvae_lower_path: ./ckpt/net_300000_lower.pth

vqvae_lower_trans_path: ./ckpt/net_300000_lower_trans.pth
use_trans: True

decay_epoch: 500

vqvae_squeeze_scale: 4 
vqvae_latent_scale: 5

vae_test_len: 32
vae_test_dim: 330
vae_test_stride: 20
vae_length: 240
vae_codebook_size: 256
vae_layer: 4
vae_grow: [1,1,2,1]
variational: False

# data c

In [46]:
!grep -n "cfg\|config\|model_config\|sc_model" /content/GestureLSM/demo2.py | head -20

23:from utils import config, logger_tools, other_tools_hf, metric, data_transfer, other_tools
178:    def __init__(self, args, cfg, ap):
225:        model_module = __import__(f"models.{cfg.model.model_name}", fromlist=["something"])
228:            getattr(model_module, cfg.model.g_name)(cfg), args.gpus
233:            logger.info(f"init {cfg.model.g_name} success")
335:        vq_configs = {
341:        for part, cfg in vq_configs.items():
342:            model = self._create_rvqvae_model(cfg['dim_pose'], part)
347:        """Create a single RVQVAE model with specified configuration."""
718:    args, cfg = config.parse_args()
724:    trainer = BaseTrainer(args, cfg, ap=audio_path)


In [47]:
# Confirm configs are correct
!grep "audio_dim\|target_length" /content/GestureLSM/configs/sc_model_config.yaml

# Then check if there's a cached .pyc or lmdb cache causing stale state
!find /content/GestureLSM -name "*.pyc" | xargs rm -f
!find /content/GestureLSM -name "test" -type d | head -10

      audio_dim: 384
      target_length: 128


In [48]:
import torch
ckpt = torch.load('/content/GestureLSM/ckpt/new_540_shortcut.bin', map_location='cpu')
state = ckpt['model_state']
for k, v in state.items():
    if 'mix_audio_text' in k:
        print(f'{k}: {v.shape}')
    if 'text_encoder_body' in k and 'weight' in k:
        print(f'{k}: {v.shape}')

module.modality_encoder.text_encoder_body.weight: torch.Size([256, 300])
module.modality_encoder.mix_audio_text.weight: torch.Size([768, 512])
module.modality_encoder.mix_audio_text.bias: torch.Size([768])


In [49]:
import shutil, os

# Delete ALL test caches so the dataloader rebuilds fresh
cache_dirs = []
for root, dirs, files in os.walk("/content/GestureLSM/outputs"):
    for d in dirs:
        if d == "test":
            cache_dirs.append(os.path.join(root, d))

# Also check tmp_dir cache
!find /tmp -name "*.lmdb" -o -name "test" -type d 2>/dev/null | head -10
!grep "tmp_dir\|cache_path" /content/GestureLSM/configs/shortcut_rvqvae_128_hf.yaml

print("Test cache dirs found:")
for d in cache_dirs:
    print(f"  {d}")

cache_path: datasets/beat_cache/beat_smplx_en_emage_2_128/
Test cache dirs found:


In [50]:
files = [
    "/content/GestureLSM/configs/sc_model_config.yaml",
    "/content/GestureLSM/configs/sc_reflow_model_config.yaml",
    "/content/GestureLSM/configs/model_config.yaml",
    "/content/GestureLSM/configs/sc_model_holistic_config.yaml",
]

for path in files:
    with open(path, "r") as f:
        content = f.read()

    changed = False
    if "audio_dim: 384" in content:
        content = content.replace("audio_dim: 384", "audio_dim: 256")
        changed = True
    if "latent_dim: 256" in content:
        content = content.replace("latent_dim: 256", "latent_dim: 768")
        changed = True

    if changed:
        with open(path, "w") as f:
            f.write(content)
        print(f"✅ Fixed: {path}")
    else:
        print(f"⚠️  Nothing to change in: {path}")

# Verify
!grep "audio_dim\|latent_dim\|target_length" /content/GestureLSM/configs/sc_model_config.yaml

✅ Fixed: /content/GestureLSM/configs/sc_model_config.yaml
✅ Fixed: /content/GestureLSM/configs/sc_reflow_model_config.yaml
✅ Fixed: /content/GestureLSM/configs/model_config.yaml
✅ Fixed: /content/GestureLSM/configs/sc_model_holistic_config.yaml
      latent_dim: 768
      audio_dim: 256
      latent_dim: 768
      target_length: 128


In [51]:
import torch
ckpt = torch.load('/content/GestureLSM/ckpt/new_540_shortcut.bin', map_location='cpu')
state = ckpt['model_state']

# Check null_cond_embed and denoiser config keys
for k, v in state.items():
    if 'null_cond' in k:
        print(f'{k}: {v.shape}')

# Check denoiser init params in checkpoint or config
!grep -A 30 "denoiser" /content/GestureLSM/configs/sc_model_config.yaml

module.denoiser.null_cond_embed: torch.Size([32, 768])
  denoiser:
    target: models.denoiser.GestureDenoiser
    params:
      input_dim: 128
      latent_dim: 768
      ff_size: 1024
      num_layers: 8
      num_heads: 4
      dropout: 0.1
      activation: "gelu"
      n_seed: 8
      flip_sin_to_cos: True
      freq_shift: 0.0
      cond_proj_dim: 256
      use_exp: ${model.use_exp}


  modality_encoder:
    target: models.layers.modality_encoder.ModalityEncoder
    params:
      data_path: ./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/
      t_fix_pre: False
      audio_dim: 256
      audio_in: 2
      raw_audio: False
      latent_dim: 768
      audio_fps: 30
      target_length: 128
      use_exp: ${model.use_exp}
  


In [52]:
!grep -n "null_cond_embed\|joint_num\|latent_dim\|seq_len" /content/GestureLSM/models/denoiser.py | head -20

14:        latent_dim=256,
25:        seq_len=32,
32:        self.latent_dim = latent_dim
39:        self.joint_num = 3 if not self.use_exp else 4
41:        self.sequence_pos_encoder = PositionalEncoding(self.latent_dim, self.dropout)
44:            CrossAttentionBlock(dim=self.latent_dim*self.joint_num,num_heads=self.num_heads,mlp_ratio=self.ff_size//self.latent_dim,drop_path=self.dropout) #hidden是对应于输入x的维度，attn_heads应该是12，这里写1是为了方便调试流程
48:            SpatialTemporalBlock(dim=self.latent_dim,num_heads=self.num_heads,mlp_ratio=self.ff_size//self.latent_dim,drop_path=self.dropout) #hidden是对应于输入x的维度，attn_heads应该是12，这里写1是为了方便调试流程
51:        self.embed_timestep = TimestepEmbedder(self.latent_dim, self.sequence_pos_encoder)
53:        self.seq_len = seq_len
56:        self.embed_text = nn.Linear(self.input_dim * self.joint_num * self.embed_context_multiplier, self.latent_dim)
58:        self.output_process = OutputProcess(self.input_dim, self.latent_dim)
60:        self.rel_pos = Sinusoida

In [53]:
files = [
    "/content/GestureLSM/configs/sc_model_config.yaml",
    "/content/GestureLSM/configs/sc_reflow_model_config.yaml",
    "/content/GestureLSM/configs/model_config.yaml",
    "/content/GestureLSM/configs/sc_model_holistic_config.yaml",
]

for path in files:
    with open(path, "r") as f:
        content = f.read()
    if "latent_dim: 768" in content:
        content = content.replace("latent_dim: 768", "latent_dim: 256")
        with open(path, "w") as f:
            f.write(content)
        print(f"✅ Reverted latent_dim: {path}")

!grep "audio_dim\|latent_dim\|target_length" /content/GestureLSM/configs/sc_model_config.yaml

✅ Reverted latent_dim: /content/GestureLSM/configs/sc_model_config.yaml
✅ Reverted latent_dim: /content/GestureLSM/configs/sc_reflow_model_config.yaml
✅ Reverted latent_dim: /content/GestureLSM/configs/model_config.yaml
✅ Reverted latent_dim: /content/GestureLSM/configs/sc_model_holistic_config.yaml
      latent_dim: 256
      audio_dim: 256
      latent_dim: 256
      target_length: 128


In [54]:
filepath = "/content/GestureLSM/models/LSM.py"
with open(filepath, "r") as f:
    content = f.read()

old = """        null_cond_embed = self.denoiser.null_cond_embed.to(at_feat.dtype)
        at_feat_uncond = null_cond_embed.unsqueeze(0).expand(batch_size, -1, -1)
        at_feat_combined = torch.cat([at_feat, at_feat_uncond], dim=0)"""

new = """        null_cond_embed = self.denoiser.null_cond_embed.to(at_feat.dtype)
        # Interpolate null_cond_embed to match at_feat's feature dimension if needed
        if null_cond_embed.shape[-1] != at_feat.shape[-1]:
            null_cond_embed = torch.nn.functional.interpolate(
                null_cond_embed.unsqueeze(0).unsqueeze(0),
                size=(null_cond_embed.shape[0], at_feat.shape[-1]),
                mode='bilinear', align_corners=False
            ).squeeze(0).squeeze(0)
        at_feat_uncond = null_cond_embed.unsqueeze(0).expand(batch_size, -1, -1)
        at_feat_combined = torch.cat([at_feat, at_feat_uncond], dim=0)"""

assert old in content, "❌ Pattern not found"
content = content.replace(old, new)
with open(filepath, "w") as f:
    f.write(content)
print("✅ Fixed apply_classifier_free_guidance")

✅ Fixed apply_classifier_free_guidance


In [55]:
import torch
ckpt = torch.load('/content/GestureLSM/ckpt/new_540_shortcut.bin', map_location='cpu')
state = ckpt['model_state']

# Check all denoiser dimension-revealing weights
for k, v in state.items():
    if any(x in k for x in ['null_cond', 'cross_attn', 'kv.', 'norm1', 'embed_style', 'mix_audio']):
        print(f'{k}: {v.shape}')

module.modality_encoder.mix_audio_text.weight: torch.Size([768, 512])
module.modality_encoder.mix_audio_text.bias: torch.Size([768])
module.denoiser.null_cond_embed: torch.Size([32, 768])
module.denoiser.cross_attn_blocks.0.norm1.weight: torch.Size([768])
module.denoiser.cross_attn_blocks.0.norm1.bias: torch.Size([768])
module.denoiser.cross_attn_blocks.0.cross_attn.q.weight: torch.Size([768, 768])
module.denoiser.cross_attn_blocks.0.cross_attn.kv.weight: torch.Size([1536, 768])
module.denoiser.cross_attn_blocks.0.cross_attn.proj.weight: torch.Size([768, 768])
module.denoiser.cross_attn_blocks.0.cross_attn.proj.bias: torch.Size([768])
module.denoiser.cross_attn_blocks.0.norm2.weight: torch.Size([768])
module.denoiser.cross_attn_blocks.0.norm2.bias: torch.Size([768])
module.denoiser.cross_attn_blocks.0.mlp.fc1.weight: torch.Size([3072, 768])
module.denoiser.cross_attn_blocks.0.mlp.fc1.bias: torch.Size([3072])
module.denoiser.cross_attn_blocks.0.mlp.fc2.weight: torch.Size([768, 3072])
mo

In [56]:
!grep -n "latent_dim\|joint_num\|hidden_size\|context_dim\|kv" /content/GestureLSM/models/denoiser.py | head -30
!grep -n "latent_dim\|context_dim\|hidden_size" /content/GestureLSM/configs/sc_model_config.yaml

14:        latent_dim=256,
32:        self.latent_dim = latent_dim
39:        self.joint_num = 3 if not self.use_exp else 4
41:        self.sequence_pos_encoder = PositionalEncoding(self.latent_dim, self.dropout)
44:            CrossAttentionBlock(dim=self.latent_dim*self.joint_num,num_heads=self.num_heads,mlp_ratio=self.ff_size//self.latent_dim,drop_path=self.dropout) #hidden是对应于输入x的维度，attn_heads应该是12，这里写1是为了方便调试流程
48:            SpatialTemporalBlock(dim=self.latent_dim,num_heads=self.num_heads,mlp_ratio=self.ff_size//self.latent_dim,drop_path=self.dropout) #hidden是对应于输入x的维度，attn_heads应该是12，这里写1是为了方便调试流程
51:        self.embed_timestep = TimestepEmbedder(self.latent_dim, self.sequence_pos_encoder)
56:        self.embed_text = nn.Linear(self.input_dim * self.joint_num * self.embed_context_multiplier, self.latent_dim)
58:        self.output_process = OutputProcess(self.input_dim, self.latent_dim)
60:        self.rel_pos = SinusoidalEmbeddings(self.latent_dim)
61:        self.input_proces

In [57]:
files = [
    "/content/GestureLSM/configs/sc_model_config.yaml",
    "/content/GestureLSM/configs/sc_reflow_model_config.yaml",
    "/content/GestureLSM/configs/model_config.yaml",
    "/content/GestureLSM/configs/sc_model_holistic_config.yaml",
]

for path in files:
    with open(path, "r") as f:
        content = f.read()

    # Only change latent_dim inside the modality_encoder params block
    # It appears AFTER "modality_encoder:" and BEFORE "denoiser:"
    if "modality_encoder:" in content and "latent_dim: 256" in content:
        # Replace only the modality_encoder's latent_dim
        # We do this by targeting the specific block
        old = """      audio_dim: 256
      audio_in: 2
      raw_audio: False
      latent_dim: 256"""
        new = """      audio_dim: 256
      audio_in: 2
      raw_audio: False
      latent_dim: 768"""
        if old in content:
            content = content.replace(old, new)
            with open(path, "w") as f:
                f.write(content)
            print(f"✅ Fixed modality_encoder latent_dim: {path}")
        else:
            print(f"⚠️  Pattern not found in: {path}")
    else:
        print(f"⚠️  Skipping: {path}")

# Verify - should show latent_dim: 768 in modality_encoder and latent_dim: 256 in denoiser
!grep -n "latent_dim" /content/GestureLSM/configs/sc_model_config.yaml

✅ Fixed modality_encoder latent_dim: /content/GestureLSM/configs/sc_model_config.yaml
✅ Fixed modality_encoder latent_dim: /content/GestureLSM/configs/sc_reflow_model_config.yaml
✅ Fixed modality_encoder latent_dim: /content/GestureLSM/configs/model_config.yaml
✅ Fixed modality_encoder latent_dim: /content/GestureLSM/configs/sc_model_holistic_config.yaml
13:      latent_dim: 256
34:      latent_dim: 768


In [58]:
!grep -rn "render\|pyrender\|trimesh\|visualize\|generate_silent" /content/GestureLSM/demo2.py | head -20
!grep -rn "generate_silent_videos\|render" /content/GestureLSM/utils/other_tools_hf.py 2>/dev/null | head -20

695:                render_vid_path = other_tools_hf.render_one_sequence_no_gt(
705:            gr.Video(value=render_vid_path, visible=True),
706:            gr.File(value=results_npz_file_save_path, label="download motion and visualize in blender"),
745:        gr.File(label="download motion and visualize in blender")
749:        2. Then, sit back and wait for the rendering to happen! This may take a while (e.g. 1-4 minutes) <br/>\
19:import utils.fast_render
464:    import pyrender
509:        mesh = pyrender.Mesh.from_trimesh(
512:        scene = pyrender.Scene()
514:        camera = pyrender.OrthographicCamera(xmag=1.0, ymag=1.0)
516:        light = pyrender.DirectionalLight(color=[1.0, 1.0, 1.0], intensity=4.0)
518:        renderer = pyrender.OffscreenRenderer(*resolution)
519:        color, _ = renderer.render(scene)
522:        renderer.delete()
585:def render_one_sequence(
622:       #import pyrender
658:    silent_video_file_path = utils.fast_render.generate_silent_videos(arg

In [59]:
# Check render settings in config
!grep -E "render_video_fps|render_video_width|render_video_height|render_concurrent_num|render_tmp_img" /content/GestureLSM/configs/shortcut_rvqvae_128_hf.yaml

# Check fast_render to understand concurrency
!cat /content/GestureLSM/utils/fast_render.py | head -80

# Check how many CPU cores are available
import os
print("CPU cores:", os.cpu_count())

# Check current GPU memory usage
!nvidia-smi --query-gpu=memory.used,memory.free,memory.total --format=csv

import os
import time
import numpy as np
import pyrender
import trimesh
import queue
import imageio
import threading
import multiprocessing
import utils.media
import glob

def deg_to_rad(degrees):
    return degrees * np.pi / 180

def create_pose_camera(angle_deg):
    angle_rad = deg_to_rad(angle_deg)
    return np.array([
        [1.0, 0.0, 0.0, 0.0],
        [0.0, np.cos(angle_rad), -np.sin(angle_rad), 1.0],
        [0.0, np.sin(angle_rad), np.cos(angle_rad), 5.0],
        [0.0, 0.0, 0.0, 1.0]
    ])

def create_pose_light(angle_deg):
    angle_rad = deg_to_rad(angle_deg)
    return np.array([
        [1.0, 0.0, 0.0, 0.0],
        [0.0, np.cos(angle_rad), -np.sin(angle_rad), 0.0],
        [0.0, np.sin(angle_rad), np.cos(angle_rad), 3.0],
        [0.0, 0.0, 0.0, 1.0]
    ])

def create_scene_with_mesh(vertices, faces, uniform_color, pose_camera, pose_light):
    trimesh_mesh = trimesh.Trimesh(vertices=vertices, faces=faces, vertex_colors=uniform_color)
    mesh = pyrender.Mesh.from_t

In [60]:
# Add this at the TOP of demo2.py, before any imports
filepath = "/content/GestureLSM/demo2.py"
with open(filepath, "r") as f:
    content = f.read()

if "PYOPENGL_PLATFORM" not in content:
    content = 'import os\nos.environ["PYOPENGL_PLATFORM"] = "egl"\n' + content
    with open(filepath, "w") as f:
        f.write(content)
    print("✅ EGL rendering enabled")
else:
    print("⚠️  Already set")

⚠️  Already set


In [61]:
!grep -rn "render_video_fps\|render_video_width\|render_video_height\|render_concurrent_num\|render_tmp_img" /content/GestureLSM/configs/
!grep -rn "render_video_fps\|render_concurrent" /content/GestureLSM/utils/config.py | head -20

302:    parser.add("--render_video_fps", default=30, type=int)
307:    parser.add("--render_concurrent_num", default=default_concurrent, type=int)


In [62]:
# Fix 2: Patch the wrong function call in fast_render.py
filepath = "/content/GestureLSM/utils/fast_render.py"
with open(filepath, "r") as f:
    content = f.read()

old = """def sub_process_process_frame_no_gt(subprocess_index, render_video_width, render_video_height, render_tmp_img_filetype, fids, frame_vertex_pairs, faces, output_dir):
    begin_ts = time.time()
    print(f"subprocess_index={subprocess_index} begin_ts={begin_ts}")

    fig_queue = queue.Queue()
    render_frames_and_enqueue(fids, frame_vertex_pairs, faces, render_video_width, render_video_height, fig_queue)"""

new = """def sub_process_process_frame_no_gt(subprocess_index, render_video_width, render_video_height, render_tmp_img_filetype, fids, frame_vertex_pairs, faces, output_dir):
    begin_ts = time.time()
    print(f"subprocess_index={subprocess_index} begin_ts={begin_ts}")

    fig_queue = queue.Queue()
    render_frames_and_enqueue_no_gt(fids, frame_vertex_pairs, faces, render_video_width, render_video_height, fig_queue)"""

assert old in content, "❌ Pattern not found"
content = content.replace(old, new)
with open(filepath, "w") as f:
    f.write(content)
print("✅ Fixed: no_gt now uses correct single-mesh renderer")

✅ Fixed: no_gt now uses correct single-mesh renderer


In [63]:
# Fix 3: Patch demo2.py to set faster render args
import os
filepath = "/content/GestureLSM/demo2.py"
with open(filepath, "r") as f:
    content = f.read()

old = "    args, cfg = config.parse_args()"
new = """    args, cfg = config.parse_args()
    args.render_video_fps = 30
    args.render_video_width = 512
    args.render_video_height = 512
    args.render_concurrent_num = os.cpu_count() or 4
    args.render_tmp_img_filetype = "jpg\""""

if "render_video_width = 512" not in content:
    assert old in content, "❌ Pattern not found"
    content = content.replace(old, new, 1)
    with open(filepath, "w") as f:
        f.write(content)
    print(f"✅ Render settings patched — concurrent_num={os.cpu_count()}")
else:
    print("⚠️  Already patched")

✅ Render settings patched — concurrent_num=2


In [64]:
# Enable GPU/EGL rendering for pyrender
filepath = "/content/GestureLSM/demo2.py"
with open(filepath, "r") as f:
    content = f.read()

if "PYOPENGL_PLATFORM" not in content:
    content = 'import os\nos.environ["PYOPENGL_PLATFORM"] = "egl"\n' + content
    with open(filepath, "w") as f:
        f.write(content)
    print("✅ EGL rendering enabled")
else:
    print("⚠️  Already set")

# Also set it in fast_render.py
filepath2 = "/content/GestureLSM/utils/fast_render.py"
with open(filepath2, "r") as f:
    content2 = f.read()

if "PYOPENGL_PLATFORM" not in content2:
    content2 = 'import os\nos.environ["PYOPENGL_PLATFORM"] = "egl"\n' + content2
    with open(filepath2, "w") as f:
        f.write(content2)
    print("✅ EGL rendering enabled in fast_render.py")
else:
    print("⚠️  Already set in fast_render.py")

⚠️  Already set
✅ EGL rendering enabled in fast_render.py


In [65]:
!pip install numpy==2.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.0/19.0 MB 91.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.4
    Uninstalling numpy-2.4.4:
      Successfully uninstalled numpy-2.4.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
whisperx 3.8.5 requires numpy>=2.1.0, but you have numpy 2.0.0 which is incompatible.
pyannote-metrics 4.0.0 requires numpy>=2.2.2, but you have numpy 2.0.0 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.2 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.2 which is incompatible.
dask-cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you ha

In [1]:
%cd /content/GestureLSM

/content/GestureLSM


In [2]:
import os
os.environ["PYOPENGL_PLATFORM"] = "egl"

import pyrender
r = pyrender.OffscreenRenderer(64, 64)

print("✅ EGL renderer works")

r.delete()

✅ EGL renderer works


In [3]:
filepath = "/content/GestureLSM/utils/fast_render.py"
with open(filepath, "r") as f:
    content = f.read()

old = """def write_images_from_queue_no_gt(fig_queue, output_dir, img_filetype):
    while True:
        e = fig_queue.get()
        if e is None:
            break
        fid, fig1, fig2 = e
        filename = os.path.join(output_dir, f"frame_{fid}.{img_filetype}")
        merged_fig = fig1 #np.hstack((fig1))"""

new = """def write_images_from_queue_no_gt(fig_queue, output_dir, img_filetype):
    while True:
        e = fig_queue.get()
        if e is None:
            break
        fid, fig1 = e
        filename = os.path.join(output_dir, f"frame_{fid}.{img_filetype}")
        merged_fig = fig1"""

assert old in content, "❌ Pattern not found"
content = content.replace(old, new)
with open(filepath, "w") as f:
    f.write(content)
print("✅ Fixed write_images_from_queue_no_gt unpacking")

✅ Fixed write_images_from_queue_no_gt unpacking


In [4]:
filepath = "/content/GestureLSM/utils/media.py"

new_content = '''import numpy as np
import subprocess
import glob
import os

def add_audio_to_video(silent_video_path, audio_path, output_video_path):
    command = [
        'ffmpeg',
        '-y',
        '-i', silent_video_path,
        '-i', audio_path,
        '-map', '0:v',
        '-map', '1:a',
        '-c:v', 'copy',
        '-shortest',
        output_video_path
    ]
    try:
        subprocess.run(command, check=True)
        print(f"Video with audio generated successfully: {output_video_path}")
    except subprocess.CalledProcessError as e:
        print(f"Error occurred: {e}")


def convert_img_to_mp4(input_pattern, output_file, framerate=30):
    # input_pattern e.g. "/path/to/frame_%d.jpg"
    # Sort frames numerically and rename to sequential before ffmpeg
    output_dir = os.path.dirname(input_pattern)
    ext = os.path.splitext(input_pattern)[1]  # .jpg or .png

    frames = sorted(
        glob.glob(os.path.join(output_dir, f"frame_*{ext}")),
        key=lambda x: int(os.path.splitext(os.path.basename(x))[0].split("_")[1])
    )

    if not frames:
        print(f"No frames found in {output_dir}")
        return

    # Rename to sequential seq_0000.jpg, seq_0001.jpg ...
    seq_pattern = os.path.join(output_dir, f"seq_%04d{ext}")
    for i, f in enumerate(frames):
        os.rename(f, os.path.join(output_dir, f"seq_{i:04d}{ext}"))

    command = [
        'ffmpeg',
        '-framerate', str(framerate),
        '-i', seq_pattern,
        '-c:v', 'libx264',
        '-pix_fmt', 'yuv420p',
        output_file,
        '-y'
    ]
    try:
        subprocess.run(command, check=True)
        print(f"Video conversion successful: {output_file}")
    except subprocess.CalledProcessError as e:
        print(f"Error during video conversion: {e}")
'''

with open(filepath, "w") as f:
    f.write(new_content)
print("✅ media.py fixed — convert_img_to_mp4 now handles non-sequential frames")

✅ media.py fixed — convert_img_to_mp4 now handles non-sequential frames


In [5]:
!grep -n "convert_img_to_mp4\|add_audio\|silence_video\|render_vid" /content/GestureLSM/utils/other_tools_hf.py | head -20

658:    silent_video_file_path = utils.fast_render.generate_silent_videos(args.render_video_fps,
659:                                                                args.render_video_width,
660:                                                                args.render_video_height,
663:                                                                int(seconds*args.render_video_fps), 
670:    utils.media.add_audio_to_video(silent_video_file_path, audio_path, final_clip)
745:    silent_video_file_path = utils.fast_render.generate_silent_videos_no_gt(args.render_video_fps,
746:                                                                args.render_video_width,
747:                                                                args.render_video_height,
750:                                                                int(seconds*args.render_video_fps), 
756:    utils.media.add_audio_to_video(silent_video_file_path, audio_path, final_clip)


In [6]:
import os

symlinks = [
    ("/content/GestureLSM/weights", "./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/weights"),
    ("/content/GestureLSM/datasets/hub/smplx_models", "./datasets/hub/smplx_models"),
    ("/content/GestureLSM/datasets/hub/pretrained_vq", "./datasets/hub/pretrained_vq"),
]

for target, link in symlinks:
    os.makedirs(os.path.dirname(os.path.abspath(link)), exist_ok=True)
    if not os.path.exists(link) and not os.path.islink(link):
        os.symlink(target, link)
        print(f"Linked: {link}")
    else:
        print(f"Already exists: {link}")

Already exists: ./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/weights
Already exists: ./datasets/hub/smplx_models
Already exists: ./datasets/hub/pretrained_vq


In [7]:
# %cd /content/GestureLSM
# !python demo2.py -c configs/shortcut_rvqvae_128_hf.yaml

In [8]:
%cd /content/GestureLSM

/content/GestureLSM


In [9]:
# Run this in Colab ONCE to fix vocab.pkl permanently
# It re-saves it with the correct module path instead of __main__
import sys, pickle

# Add Vocab to sys.modules so pickle can find it
sys.path.insert(0, "/content/GestureLSM")
from dataloaders.build_vocab import Vocab
import __main__
__main__.Vocab = Vocab  # inject so existing pkl loads

vocab_path = "./datasets/BEAT_SMPL/beat_v2.0.0/beat_english_v2.0.0/weights/vocab.pkl"

# Load with the patched __main__
with open(vocab_path, "rb") as f:
    lang_model = pickle.load(f)

# Re-save with explicit module so it works from any entry point
import dataloaders.build_vocab as bv
lang_model.__class__ = bv.Vocab  # rebind class to proper module

with open(vocab_path, "wb") as f:
    pickle.dump(lang_model, f)

print("✅ vocab.pkl re-saved with proper module path")

✅ vocab.pkl re-saved with proper module path


In [10]:
%%writefile /content/GestureLSM/utils/fast_render.py
import os
import time
import numpy as np
import pyrender
import trimesh
import queue
import imageio
import threading
import multiprocessing
import glob
import subprocess

os.environ["PYOPENGL_PLATFORM"] = "egl"

def deg_to_rad(degrees):
    return degrees * np.pi / 180

def create_pose_camera(angle_deg):
    angle_rad = deg_to_rad(angle_deg)
    return np.array([
        [1.0, 0.0, 0.0, 0.0],
        [0.0, np.cos(angle_rad), -np.sin(angle_rad), 1.0],
        [0.0, np.sin(angle_rad), np.cos(angle_rad), 5.0],
        [0.0, 0.0, 0.0, 1.0]
    ])

def create_pose_light(angle_deg):
    angle_rad = deg_to_rad(angle_deg)
    return np.array([
        [1.0, 0.0, 0.0, 0.0],
        [0.0, np.cos(angle_rad), -np.sin(angle_rad), 0.0],
        [0.0, np.sin(angle_rad), np.cos(angle_rad), 3.0],
        [0.0, 0.0, 0.0, 1.0]
    ])

def create_scene_with_mesh(vertices, faces, uniform_color, pose_camera, pose_light):
    trimesh_mesh = trimesh.Trimesh(vertices=vertices, faces=faces, vertex_colors=uniform_color)
    mesh = pyrender.Mesh.from_trimesh(trimesh_mesh, smooth=True)
    scene = pyrender.Scene()
    scene.add(mesh)
    camera = pyrender.OrthographicCamera(xmag=1.0, ymag=1.0)
    scene.add(camera, pose=pose_camera)
    light = pyrender.DirectionalLight(color=[1.0, 1.0, 1.0], intensity=4.0)
    scene.add(light, pose=pose_light)
    return scene

def do_render_one_frame(renderer, frame_idx, vertices, vertices1, faces):
    if frame_idx % 100 == 0:
        print('processed', frame_idx, 'frames')
    uniform_color = [220, 220, 220, 255]
    pose_camera = create_pose_camera(angle_deg=-2)
    pose_light  = create_pose_light(angle_deg=-30)
    figs = []
    for vtx in [vertices, vertices1]:
        scene = create_scene_with_mesh(vtx, faces, uniform_color, pose_camera, pose_light)
        fig, _ = renderer.render(scene)
        figs.append(fig)
    return figs[0], figs[1]

def do_render_one_frame_no_gt(renderer, frame_idx, vertices, faces):
    if frame_idx % 100 == 0:
        print('processed', frame_idx, 'frames')
    uniform_color = [220, 220, 220, 255]
    pose_camera = create_pose_camera(angle_deg=-2)
    pose_light  = create_pose_light(angle_deg=-30)
    scene = create_scene_with_mesh(vertices, faces, uniform_color, pose_camera, pose_light)
    fig, _ = renderer.render(scene)
    return fig

def write_images_from_queue(fig_queue, output_dir, img_filetype):
    while True:
        e = fig_queue.get()
        if e is None:
            break
        fid, fig1, fig2 = e
        filename = os.path.join(output_dir, f"frame_{fid}.{img_filetype}")
        merged_fig = np.hstack((fig1, fig2))
        try:
            imageio.imwrite(filename, merged_fig)
        except Exception as ex:
            print(f"Error writing image {filename}: {ex}")
            raise ex

def write_images_from_queue_no_gt(fig_queue, output_dir, img_filetype):
    while True:
        e = fig_queue.get()
        if e is None:
            break
        fid, fig1 = e   # ← fixed: only 2 values for no_gt
        filename = os.path.join(output_dir, f"frame_{fid}.{img_filetype}")
        try:
            imageio.imwrite(filename, fig1)
        except Exception as ex:
            print(f"Error writing image {filename}: {ex}")
            raise ex

def render_frames_and_enqueue(fids, frame_vertex_pairs, faces, render_width, render_height, fig_queue):
    fig_resolution = (render_width // 2, render_height)
    renderer = pyrender.OffscreenRenderer(*fig_resolution)
    for idx, fid in enumerate(fids):
        fig1, fig2 = do_render_one_frame(renderer, fid, frame_vertex_pairs[idx][0], frame_vertex_pairs[idx][1], faces)
        fig_queue.put((fid, fig1, fig2))
    renderer.delete()

def render_frames_and_enqueue_no_gt(fids, frame_vertex_pairs, faces, render_width, render_height, fig_queue):
    fig_resolution = (render_width // 2, render_height)
    renderer = pyrender.OffscreenRenderer(*fig_resolution)
    for idx, fid in enumerate(fids):
        fig1 = do_render_one_frame_no_gt(renderer, fid, frame_vertex_pairs[idx][0], faces)
        fig_queue.put((fid, fig1))   # ← fixed: only 2 values
    renderer.delete()

def sub_process_process_frame(subprocess_index, render_video_width, render_video_height,
                               render_tmp_img_filetype, fids, frame_vertex_pairs, faces, output_dir):
    begin_ts = time.time()
    print(f"subprocess_index={subprocess_index} begin_ts={begin_ts}")
    fig_queue = queue.Queue()
    render_frames_and_enqueue(fids, frame_vertex_pairs, faces, render_video_width, render_video_height, fig_queue)
    fig_queue.put(None)
    render_end_ts = time.time()
    image_writer_thread = threading.Thread(
        target=write_images_from_queue,
        args=(fig_queue, output_dir, render_tmp_img_filetype))
    image_writer_thread.start()
    image_writer_thread.join()
    write_end_ts = time.time()
    print(f"subprocess_index={subprocess_index} render={render_end_ts-begin_ts:.2f} all={write_end_ts-begin_ts:.2f}")

def sub_process_process_frame_no_gt(subprocess_index, render_video_width, render_video_height,
                                     render_tmp_img_filetype, fids, frame_vertex_pairs, faces, output_dir):
    begin_ts = time.time()
    print(f"subprocess_index={subprocess_index} begin_ts={begin_ts}")
    fig_queue = queue.Queue()
    render_frames_and_enqueue_no_gt(fids, frame_vertex_pairs, faces,   # ← fixed: no_gt version
                                     render_video_width, render_video_height, fig_queue)
    fig_queue.put(None)
    render_end_ts = time.time()
    image_writer_thread = threading.Thread(
        target=write_images_from_queue_no_gt,
        args=(fig_queue, output_dir, render_tmp_img_filetype))
    image_writer_thread.start()
    image_writer_thread.join()
    write_end_ts = time.time()
    print(f"subprocess_index={subprocess_index} render={render_end_ts-begin_ts:.2f} all={write_end_ts-begin_ts:.2f}")

def distribute_frames(frames, render_video_fps, render_concurent_nums, vertices_all, vertices1_all):
    sample_interval = max(1, int(30 // render_video_fps))
    subproc_frame_ids = [[] for _ in range(render_concurent_nums)]
    subproc_vertices  = [[] for _ in range(render_concurent_nums)]
    sampled_frame_id  = 0
    for i in range(frames):
        if i % sample_interval != 0:
            continue
        subprocess_index = sampled_frame_id % render_concurent_nums
        subproc_frame_ids[subprocess_index].append(sampled_frame_id)
        subproc_vertices[subprocess_index].append((vertices_all[i], vertices1_all[i]))
        sampled_frame_id += 1
    return subproc_frame_ids, subproc_vertices

def distribute_frames_no_gt(frames, render_video_fps, render_concurent_nums, vertices_all):
    sample_interval = max(1, int(30 // render_video_fps))
    subproc_frame_ids = [[] for _ in range(render_concurent_nums)]
    subproc_vertices  = [[] for _ in range(render_concurent_nums)]
    sampled_frame_id  = 0
    for i in range(frames):
        if i % sample_interval != 0:
            continue
        subprocess_index = sampled_frame_id % render_concurent_nums
        subproc_frame_ids[subprocess_index].append(sampled_frame_id)
        subproc_vertices[subprocess_index].append((vertices_all[i], vertices_all[i]))
        sampled_frame_id += 1
    return subproc_frame_ids, subproc_vertices


# ── Fixed video assembly (sorts frames numerically before ffmpeg) ─────────────

def _frames_to_mp4(output_dir, ext, output_file, fps):
    """Sort frame_N.ext numerically → seq_0000.ext → ffmpeg → mp4."""
    frames = sorted(
        glob.glob(os.path.join(output_dir, f"frame_*{ext}")),
        key=lambda x: int(os.path.splitext(os.path.basename(x))[0].split("_")[1])
    )
    if not frames:
        raise FileNotFoundError(f"No frames found in {output_dir}")

    # Rename to sequential
    for i, f in enumerate(frames):
        os.rename(f, os.path.join(output_dir, f"seq_{i:04d}{ext}"))

    seq_pattern = os.path.join(output_dir, f"seq_%04d{ext}")
    r = subprocess.run([
        "ffmpeg", "-y", "-framerate", str(fps),
        "-i", seq_pattern,
        "-c:v", "libx264", "-pix_fmt", "yuv420p",
        output_file
    ], capture_output=True, text=True)

    # Clean up seq frames regardless
    for f in glob.glob(os.path.join(output_dir, f"seq_*{ext}")):
        os.remove(f)

    if r.returncode != 0:
        raise RuntimeError(f"ffmpeg failed:\n{r.stderr}")

    print(f"Video conversion successful: {output_file}")
    return output_file


def generate_silent_videos(render_video_fps, render_video_width, render_video_height,
                            render_concurent_nums, render_tmp_img_filetype,
                            frames, vertices_all, vertices1_all, faces, output_dir):

    subproc_frame_ids, subproc_vertices = distribute_frames(
        frames, render_video_fps, render_concurent_nums, vertices_all, vertices1_all)

    print(f"generate_silent_videos concurrentNum={render_concurent_nums} time={time.time()}")
    with multiprocessing.Pool(render_concurent_nums) as pool:
        pool.starmap(sub_process_process_frame, [
            (i, render_video_width, render_video_height, render_tmp_img_filetype,
             subproc_frame_ids[i], subproc_vertices[i], faces, output_dir)
            for i in range(render_concurent_nums)
        ])

    output_file = os.path.join(output_dir, "silence_video.mp4")
    _frames_to_mp4(output_dir, f".{render_tmp_img_filetype}", output_file, render_video_fps)
    return output_file


def generate_silent_videos_no_gt(render_video_fps, render_video_width, render_video_height,
                                   render_concurent_nums, render_tmp_img_filetype,
                                   frames, vertices_all, faces, output_dir):

    subproc_frame_ids, subproc_vertices = distribute_frames_no_gt(
        frames, render_video_fps, render_concurent_nums, vertices_all)

    print(f"generate_silent_videos concurrentNum={render_concurent_nums} time={time.time()}")
    with multiprocessing.Pool(render_concurent_nums) as pool:
        pool.starmap(sub_process_process_frame_no_gt, [
            (i, render_video_width, render_video_height, render_tmp_img_filetype,
             subproc_frame_ids[i], subproc_vertices[i], faces, output_dir)
            for i in range(render_concurent_nums)
        ])

    output_file = os.path.join(output_dir, "silence_video.mp4")
    _frames_to_mp4(output_dir, f".{render_tmp_img_filetype}", output_file, render_video_fps)
    return output_file

Overwriting /content/GestureLSM/utils/fast_render.py


In [11]:
!pip install numpy==2.0

In [12]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 113.5 MB/s eta 0:00:00


In [13]:
filepath = "/content/GestureLSM/utils/other_tools_hf.py"
with open(filepath, "r") as f:
    content = f.read()

# Find and add prints around the render call
old = """    silent_video_file_path = utils.fast_render.generate_silent_videos_no_gt(args.render_video_fps,
                                                                args.render_video_width,
                                                                args.render_video_height,
                                                                args.render_concurrent_num,
                                                                args.render_tmp_img_filetype,
                                                                int(seconds*args.render_video_fps),"""

new = """    print("[DEBUG] Starting generate_silent_videos_no_gt...", flush=True)
    silent_video_file_path = utils.fast_render.generate_silent_videos_no_gt(args.render_video_fps,
                                                                args.render_video_width,
                                                                args.render_video_height,
                                                                args.render_concurrent_num,
                                                                args.render_tmp_img_filetype,
                                                                int(seconds*args.render_video_fps),"""

if old in content:
    content = content.replace(old, new)
    with open(filepath, "w") as f:
        f.write(content)
    print("✅ Debug print added before generate_silent_videos_no_gt")
else:
    print("❌ Pattern not found — show me the render call in other_tools_hf.py:")
    # Find it manually
    for i, line in enumerate(content.split('\n')):
        if 'generate_silent_videos_no_gt' in line or 'add_audio' in line or 'silent_video' in line:
            print(f"  {i}: {line}")

✅ Debug print added before generate_silent_videos_no_gt


In [14]:
filepath = "/content/GestureLSM/utils/fast_render.py"
with open(filepath, "r") as f:
    content = f.read()

old = "    output_file = os.path.join(output_dir, \"silence_video.mp4\")\n    _frames_to_mp4(output_dir"
new = "    print('[DEBUG] All threads done, starting _frames_to_mp4...', flush=True)\n    output_file = os.path.join(output_dir, \"silence_video.mp4\")\n    _frames_to_mp4(output_dir"

if old in content:
    content = content.replace(old, new, 1)  # only first occurrence (no_gt version)
    with open(filepath, "w") as f:
        f.write(content)
    print("✅ Debug print added before _frames_to_mp4")
else:
    print("❌ Pattern not found")

✅ Debug print added before _frames_to_mp4


In [15]:
filepath = "/content/GestureLSM/utils/fast_render.py"
with open(filepath, "r") as f:
    content = f.read()

# Replace Pool with threads for no_gt version
old = """    print(f"generate_silent_videos concurrentNum={render_concurent_nums} time={time.time()}")
    with multiprocessing.Pool(render_concurent_nums) as pool:
        pool.starmap(sub_process_process_frame_no_gt, [
            (i, render_video_width, render_video_height, render_tmp_img_filetype,
             subproc_frame_ids[i], subproc_vertices[i], faces, output_dir)
            for i in range(render_concurent_nums)
        ])"""

new = """    print(f"generate_silent_videos concurrentNum={render_concurent_nums} time={time.time()}")
    import threading
    threads = []
    for i in range(render_concurent_nums):
        t = threading.Thread(
            target=sub_process_process_frame_no_gt,
            args=(i, render_video_width, render_video_height, render_tmp_img_filetype,
                  subproc_frame_ids[i], subproc_vertices[i], faces, output_dir)
        )
        t.start()
        threads.append(t)
    for t in threads:
        t.join()
    print('[DEBUG] All threads joined', flush=True)"""

if old in content:
    content = content.replace(old, new)
    with open(filepath, "w") as f:
        f.write(content)
    print("✅ Replaced multiprocessing.Pool with threads in generate_silent_videos_no_gt")
else:
    print("❌ Pattern not found — showing generate_silent_videos_no_gt:")
    start = content.find("def generate_silent_videos_no_gt")
    print(content[start:start+600])

✅ Replaced multiprocessing.Pool with threads in generate_silent_videos_no_gt


In [16]:
filepath = "/content/GestureLSM/utils/fast_render.py"
with open(filepath, "r") as f:
    content = f.read()

old = """    print(f"generate_silent_videos concurrentNum={render_concurent_nums} time={time.time()}")
    import threading
    threads = []
    for i in range(render_concurent_nums):
        t = threading.Thread(
            target=sub_process_process_frame_no_gt,
            args=(i, render_video_width, render_video_height, render_tmp_img_filetype,
                  subproc_frame_ids[i], subproc_vertices[i], faces, output_dir)
        )
        t.start()
        threads.append(t)
    for t in threads:
        t.join()
    print('[DEBUG] All threads joined', flush=True)"""

new = """    print(f"generate_silent_videos concurrentNum={render_concurent_nums} time={time.time()}")
    # Run sequentially in same process — avoids multiprocessing/thread EGL context issues
    for i in range(render_concurent_nums):
        sub_process_process_frame_no_gt(
            i, render_video_width, render_video_height, render_tmp_img_filetype,
            subproc_frame_ids[i], subproc_vertices[i], faces, output_dir)
    print('[DEBUG] All frames rendered', flush=True)"""

if old in content:
    content = content.replace(old, new)
    with open(filepath, "w") as f:
        f.write(content)
    print("✅ Replaced threads with sequential rendering")
else:
    print("❌ Pattern not found — current generate_silent_videos_no_gt:")
    start = content.find("def generate_silent_videos_no_gt")
    print(content[start:start+800])

✅ Replaced threads with sequential rendering


In [17]:
%cd /content/GestureLSM

/content/GestureLSM


In [18]:
%%writefile app.py
import os, sys, glob, time, shutil, subprocess, tempfile, warnings
os.environ["PYOPENGL_PLATFORM"] = "egl"
sys.path.insert(0, "/content/GestureLSM")

import streamlit as st
import numpy as np
import soundfile as sf

import __main__
from dataloaders.build_vocab import Vocab
__main__.Vocab = Vocab

st.set_page_config(page_title="GestureLSM", page_icon="🤸", layout="centered")
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@300;400;600&display=swap');
html,body,[class*="css"]{font-family:'DM Sans',sans-serif;background:#0a0a0f;color:#e8e8f0;}
h1,h2,h3{font-family:'Space Mono',monospace;}
.stApp{background:#0a0a0f;}
.hero{text-align:center;padding:2.5rem 0 1.5rem;}
.hero h1{font-size:2.2rem;background:linear-gradient(135deg,#a78bfa,#60a5fa,#34d399);
  -webkit-background-clip:text;-webkit-text-fill-color:transparent;margin-bottom:.4rem;}
.hero p{color:#888;font-size:.95rem;}
.sbox{background:#13131f;border:1px solid #2a2a3f;border-radius:10px;
  padding:.8rem 1.2rem;margin:.5rem 0;font-family:'Space Mono',monospace;
  font-size:.78rem;color:#a78bfa;}
.sbox.ok{border-color:#34d399;color:#34d399;}
.sbox.err{border-color:#f87171;color:#f87171;}
.sbox.run{border-color:#60a5fa;color:#60a5fa;}
div[data-testid="stFileUploader"]{background:#13131f;border:2px dashed #2a2a3f;border-radius:12px;padding:1rem;}
.stButton>button{width:100%;background:linear-gradient(135deg,#7c3aed,#2563eb);color:white;
  border:none;border-radius:8px;padding:.7rem 2rem;font-family:'Space Mono',monospace;
  font-size:.85rem;font-weight:700;letter-spacing:.05em;}
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="hero">
  <h1>GestureLSM</h1>
  <p>Speech-to-Gesture Generation · Upload audio, get animated motion</p>
</div>""", unsafe_allow_html=True)


def slog(area, msg, kind=""):
    icons = {"ok":"✓","err":"✗","run":"▶","":"◆"}
    area.markdown(f'<div class="sbox {kind}">{icons.get(kind,"◆")} {msg}</div>',
                  unsafe_allow_html=True)


def frames_to_video(frames_dir, ext, output_mp4, fps=30):
    """Sort frames numerically, rename sequentially, assemble with ffmpeg."""
    frames = sorted(
        glob.glob(os.path.join(frames_dir, f"frame_*{ext}")),
        key=lambda x: int(os.path.splitext(os.path.basename(x))[0].split("_")[1])
    )
    if not frames:
        raise FileNotFoundError(f"No frames found in {frames_dir}")
    for i, f in enumerate(frames):
        os.rename(f, os.path.join(frames_dir, f"seq_{i:04d}{ext}"))
    r = subprocess.run([
        "ffmpeg", "-y", "-framerate", str(fps),
        "-i", os.path.join(frames_dir, f"seq_%04d{ext}"),
        "-c:v", "libx264", "-pix_fmt", "yuv420p", output_mp4
    ], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"ffmpeg silent video failed:\n{r.stderr}")
    for f in glob.glob(os.path.join(frames_dir, f"seq_*{ext}")):
        os.remove(f)


def add_audio_to_video(silent_mp4, audio_wav, final_mp4):
    r = subprocess.run([
        "ffmpeg", "-y", "-i", silent_mp4, "-i", audio_wav,
        "-map", "0:v", "-map", "1:a", "-c:v", "copy", "-shortest", final_mp4
    ], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"ffmpeg audio merge failed:\n{r.stderr}")


def patch_fast_render():
    """Monkey-patch generate_silent_videos_no_gt to use fixed frame assembly."""
    import utils.fast_render as fr

    def fixed_gen_no_gt(fps, w, h, concurrent, ext, n_frames,
                        vertices_all, faces, output_dir):
        subproc_ids, subproc_verts = fr.distribute_frames_no_gt(
            n_frames, fps, concurrent, vertices_all)
        import multiprocessing
        print(f"generate_silent_videos concurrentNum={concurrent} time={time.time()}")
        with multiprocessing.Pool(concurrent) as pool:
            pool.starmap(fr.sub_process_process_frame_no_gt, [
                (i, w, h, ext, subproc_ids[i], subproc_verts[i], faces, output_dir)
                for i in range(concurrent)
            ])
        silent_mp4 = os.path.join(output_dir, "silence_video.mp4")
        frames_to_video(output_dir, f".{ext}", silent_mp4, fps)
        return silent_mp4

    fr.generate_silent_videos_no_gt = fixed_gen_no_gt


@st.cache_resource(show_spinner=False)
def get_args_cfg():
    _orig = sys.argv[:]
    sys.argv = ["app.py", "--config", "configs/shortcut_rvqvae_128_hf.yaml"]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        from utils import config
        args, cfg = config.parse_args()
    sys.argv = _orig
    args.render_video_fps        = 30
    args.render_video_width      = 512
    args.render_video_height     = 512
    args.render_concurrent_num   = os.cpu_count() or 2
    args.render_tmp_img_filetype = "jpg"
    return args, cfg


def run_pipeline(audio_bytes, slots):
    slog(slots[0], "Parsing config…", "run")
    args, cfg = get_args_cfg()
    slog(slots[0], "Config ready ✓", "ok")

    # Save audio to temp file
    tmp_dir    = tempfile.mkdtemp(prefix="glsm_")
    audio_path = os.path.join(tmp_dir, "input.wav")
    with open(audio_path, "wb") as f:
        f.write(audio_bytes)
    data, sr = sf.read(audio_path)
    if data.ndim > 1:
        data = data[:, 0]
    ap = (sr, (data * 32767).astype(np.int16))

    slog(slots[1], "Importing GestureLSM modules…", "run")
    _orig = sys.argv[:]
    sys.argv = ["demo2.py", "--config", "configs/shortcut_rvqvae_128_hf.yaml"]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        import demo2 as d2
    sys.argv = _orig
    slog(slots[1], "Import done ✓", "ok")

    slog(slots[2], "WhisperX alignment + building dataloader…", "run")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        trainer = d2.BaseTrainer(args, cfg, ap=ap)
    from utils import other_tools
    other_tools.load_checkpoints(trainer.model, args.test_ckpt, args.g_name)
    slog(slots[2], "Model loaded ✓", "ok")

    # Patch BEFORE test_demo so rendering uses fixed assembly
    patch_fast_render()

    slog(slots[3], "Inference + rendering (~2 min)…", "run")
    result     = trainer.test_demo(999)
    render_vid = result[0].value if hasattr(result[0], "value") else result[0]
    npz_path   = result[1].value if hasattr(result[1], "value") else result[1]
    slog(slots[3], "Inference done ✓", "ok")

    # Merge audio
    slog(slots[4], "Merging audio…", "run")
    if render_vid and os.path.exists(render_vid):
        final_vid = render_vid.replace(".mp4", "_final.mp4")
        wav_path  = getattr(trainer, "audio_path", None)
        if wav_path and os.path.exists(wav_path):
            try:
                add_audio_to_video(render_vid, wav_path, final_vid)
                render_vid = final_vid
                slog(slots[4], "Audio merged ✓", "ok")
            except Exception as e:
                slog(slots[4], f"Audio merge skipped: {e}", "err")
        else:
            slog(slots[4], "No audio file found — returning silent video", "err")

    return render_vid, npz_path


# ── UI ─────────────────────────────────────────────────────────────────────────
uploaded = st.file_uploader(
    "Upload speech audio (WAV / MP3 / FLAC)", type=["wav","mp3","flac","ogg"])

# st.markdown("**Or try an example:**")
# examples = [
#     "demo/examples/2_scott_0_1_1.wav",
#     "demo/examples/2_scott_0_2_2.wav",
#     "demo/examples/2_scott_0_3_3.wav",
# ]
# cols = st.columns(len(examples))
# example_chosen = None
# for i, (col, ex) in enumerate(zip(cols, examples)):
#     with col:
#         if st.button(f"Example {i+1}", key=f"ex{i}"):
#             example_chosen = ex

st.markdown("---")
run_btn = st.button("▶  Generate Gesture Video")

slots      = [st.empty() for _ in range(5)]
vid_area   = st.empty()
dl_area    = st.empty()

if run_btn:
    audio_bytes = None
    if uploaded:
        audio_bytes = uploaded.read()
    elif example_chosen:
        ex_full = os.path.join("/content/GestureLSM", example_chosen)
        if os.path.exists(ex_full):
            with open(ex_full, "rb") as f:
                audio_bytes = f.read()

    if not audio_bytes:
        st.warning("Please upload an audio file or choose an example.")
    else:
        try:
            vid_path, npz_path = run_pipeline(audio_bytes, slots)
            if vid_path and os.path.exists(vid_path):
                with vid_area:
                    st.video(vid_path)
                with dl_area:
                    c1, c2 = st.columns(2)
                    with c1:
                        with open(vid_path, "rb") as vf:
                            st.download_button("⬇ Download Video", vf.read(),
                                               "gesture_output.mp4", "video/mp4")
                    with c2:
                        if npz_path and os.path.exists(npz_path):
                            with open(npz_path, "rb") as nf:
                                st.download_button("⬇ Download Motion (.npz)",
                                                   nf.read(), "motion.npz")
            else:
                st.error("Video not found — check the status messages above.")
        except Exception as e:
            import traceback
            st.error(str(e))
            st.code(traceback.format_exc())

Writing app.py


In [19]:
filepath = "/content/GestureLSM/app.py"
with open(filepath, "r") as f:
    content = f.read()

old = """    render_vid = result[0].value if hasattr(result[0], "value") else result[0]
    npz_path   = result[1].value if hasattr(result[1], "value") else result[1]"""

new = """    # result can be gr.Video object, dict, or plain string — extract path safely
    def extract_path(r):
        if isinstance(r, str):
            return r
        if isinstance(r, dict):
            return r.get("value") or r.get("path") or r.get("name")
        if hasattr(r, "value"):
            return r.value
        return str(r) if r else None

    render_vid = extract_path(result[0])
    npz_path   = extract_path(result[1])
    print(f"[DEBUG] render_vid={render_vid}, npz_path={npz_path}", flush=True)"""

assert old in content, "❌ Pattern not found"
content = content.replace(old, new)
with open(filepath, "w") as f:
    f.write(content)
print("✅ Fixed path extraction")

✅ Fixed path extraction


In [20]:
filepath = "/content/GestureLSM/app.py"
with open(filepath, "r") as f:
    content = f.read()

old = """    print(f"[DEBUG] render_vid={render_vid}, npz_path={npz_path}", flush=True)"""

new = """    print(f"[DEBUG] render_vid={render_vid}, npz_path={npz_path}", flush=True)

    # Fallback: find latest mp4/npz in outputs if path extraction failed
    if not render_vid or not os.path.exists(str(render_vid)):
        import glob as _glob
        mp4s = sorted(_glob.glob("/content/GestureLSM/outputs/audio2pose/custom/**/*.mp4",
                                  recursive=True), key=os.path.getmtime)
        render_vid = mp4s[-1] if mp4s else None
        print(f"[DEBUG] fallback render_vid={render_vid}", flush=True)
    if not npz_path or not os.path.exists(str(npz_path)):
        import glob as _glob
        npzs = sorted(_glob.glob("/content/GestureLSM/outputs/audio2pose/custom/**/*.npz",
                                  recursive=True), key=os.path.getmtime)
        npz_path = npzs[-1] if npzs else None"""

assert old in content, "❌ Pattern not found"
content = content.replace(old, new)
with open(filepath, "w") as f:
    f.write(content)
print("✅ Added fallback path finder")

✅ Added fallback path finder


In [21]:
filepath = "/content/GestureLSM/app.py"
with open(filepath, "r") as f:
    content = f.read()

old = """    if not audio_bytes:
        st.warning("Please upload an audio file or choose an example.")
    else:
        try:
            vid_path, npz_path = run_pipeline(audio_bytes, slots)"""

new = """    if not audio_bytes:
        st.warning("Please upload an audio file or choose an example.")
    else:
        try:
            import threading
            result_container = {}
            error_container  = {}

            def _run():
                try:
                    result_container["vid"], result_container["npz"] = run_pipeline(audio_bytes, slots)
                except Exception as ex:
                    import traceback
                    error_container["err"] = str(ex)
                    error_container["tb"]  = traceback.format_exc()

            t = threading.Thread(target=_run, daemon=True)
            t.start()

            # Poll until done — keeps Streamlit alive
            poll_slot = st.empty()
            dots = 0
            while t.is_alive():
                dots = (dots + 1) % 4
                poll_slot.markdown(f"⏳ Running{'.' * (dots+1)}")
                time.sleep(3)
            poll_slot.empty()
            t.join()

            if error_container:
                st.error(error_container["err"])
                st.code(error_container["tb"])
                st.stop()

            vid_path  = result_container.get("vid")
            npz_path  = result_container.get("npz")"""

assert old in content, "❌ Pattern not found"
content = content.replace(old, new)

# Fix the indentation of the rest of the block (add 4 spaces)
old2 = """            if vid_path and os.path.exists(vid_path):
                with vid_area:
                    st.video(vid_path)
                with dl_area:
                    c1, c2 = st.columns(2)
                    with c1:
                        with open(vid_path, "rb") as vf:
                            st.download_button("⬇ Download Video", vf.read(),
                                               "gesture_output.mp4", "video/mp4")
                    with c2:
                        if npz_path and os.path.exists(npz_path):
                            with open(npz_path, "rb") as nf:
                                st.download_button("⬇ Download Motion (.npz)",
                                                   nf.read(), "motion.npz")
            else:
                st.error("Video not found — check the status messages above.")
        except Exception as e:
            import traceback
            st.error(str(e))
            st.code(traceback.format_exc())"""

new2 = """            if vid_path and os.path.exists(str(vid_path)):
                with vid_area:
                    st.video(str(vid_path))
                with dl_area:
                    c1, c2 = st.columns(2)
                    with c1:
                        with open(vid_path, "rb") as vf:
                            st.download_button("⬇ Download Video", vf.read(),
                                               "gesture_output.mp4", "video/mp4")
                    with c2:
                        if npz_path and os.path.exists(str(npz_path)):
                            with open(npz_path, "rb") as nf:
                                st.download_button("⬇ Download Motion (.npz)",
                                                   nf.read(), "motion.npz")
            else:
                st.error("Video not found — check the status messages above.")
        except Exception as e:
            import traceback
            st.error(str(e))
            st.code(traceback.format_exc())"""

assert old2 in content, "❌ Pattern 2 not found"
content = content.replace(old2, new2)

with open(filepath, "w") as f:
    f.write(content)
print("✅ Pipeline now runs in background thread with polling")

✅ Pipeline now runs in background thread with polling


In [22]:
%%writefile app.py
"""
GestureLSM Streamlit App
Run from /content/GestureLSM:
    streamlit run app.py --server.port 8501
"""
import os, sys, glob, time, shutil, subprocess, tempfile, warnings, queue, threading
os.environ["PYOPENGL_PLATFORM"] = "egl"
sys.path.insert(0, "/content/GestureLSM")

import streamlit as st
import numpy as np
import soundfile as sf

st.set_page_config(page_title="GestureLSM", page_icon="🤸", layout="centered")
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@300;400;600&display=swap');
html,body,[class*="css"]{font-family:'DM Sans',sans-serif;background:#0a0a0f;color:#e8e8f0;}
h1,h2,h3{font-family:'Space Mono',monospace;}
.stApp{background:#0a0a0f;}
.hero{text-align:center;padding:2.5rem 0 1.5rem;}
.hero h1{font-size:2.2rem;background:linear-gradient(135deg,#a78bfa,#60a5fa,#34d399);
  -webkit-background-clip:text;-webkit-text-fill-color:transparent;margin-bottom:.4rem;}
.hero p{color:#888;font-size:.95rem;}
.sbox{background:#13131f;border:1px solid #2a2a3f;border-radius:10px;
  padding:.8rem 1.2rem;margin:.4rem 0;font-family:'Space Mono',monospace;font-size:.78rem;color:#a78bfa;}
.sbox.ok{border-color:#34d399;color:#34d399;}
.sbox.err{border-color:#f87171;color:#f87171;}
.sbox.run{border-color:#60a5fa;color:#60a5fa;}
div[data-testid="stFileUploader"]{background:#13131f;border:2px dashed #2a2a3f;border-radius:12px;padding:1rem;}
.stButton>button{width:100%;background:linear-gradient(135deg,#7c3aed,#2563eb);color:white;
  border:none;border-radius:8px;padding:.7rem 2rem;font-family:'Space Mono',monospace;
  font-size:.85rem;font-weight:700;letter-spacing:.05em;}
</style>
""", unsafe_allow_html=True)

st.markdown("""
<div class="hero">
  <h1>GestureLSM</h1>
  <p>Speech-to-Gesture Generation · Upload audio, get animated motion</p>
</div>""", unsafe_allow_html=True)


# ── Helpers ───────────────────────────────────────────────────────────────────

def render_log(msg, kind=""):
    icons = {"ok":"✓","err":"✗","run":"▶","":"◆"}
    return f'<div class="sbox {kind}">{icons.get(kind,"◆")} {msg}</div>'


def frames_to_video(frames_dir, ext, output_mp4, fps=30):
    frames = sorted(
        glob.glob(os.path.join(frames_dir, f"frame_*{ext}")),
        key=lambda x: int(os.path.splitext(os.path.basename(x))[0].split("_")[1])
    )
    if not frames:
        raise FileNotFoundError(f"No frames found in {frames_dir}")
    for i, f in enumerate(frames):
        os.rename(f, os.path.join(frames_dir, f"seq_{i:04d}{ext}"))
    r = subprocess.run([
        "ffmpeg", "-y", "-framerate", str(fps),
        "-i", os.path.join(frames_dir, f"seq_%04d{ext}"),
        "-c:v", "libx264", "-pix_fmt", "yuv420p", output_mp4
    ], capture_output=True, text=True)
    for f in glob.glob(os.path.join(frames_dir, f"seq_*{ext}")):
        os.remove(f)
    if r.returncode != 0:
        raise RuntimeError(f"ffmpeg failed:\n{r.stderr}")


def add_audio_to_video(silent_mp4, audio_wav, final_mp4):
    r = subprocess.run([
        "ffmpeg", "-y", "-i", silent_mp4, "-i", audio_wav,
        "-map", "0:v", "-map", "1:a", "-c:v", "copy", "-shortest", final_mp4
    ], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"ffmpeg audio merge failed:\n{r.stderr}")


@st.cache_resource(show_spinner=False)
def get_args_cfg():
    _orig = sys.argv[:]
    sys.argv = ["app.py", "--config", "configs/shortcut_rvqvae_128_hf.yaml"]
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        from utils import config
        args, cfg = config.parse_args()
    sys.argv = _orig
    args.render_video_fps        = 30
    args.render_video_width      = 512
    args.render_video_height     = 512
    args.render_concurrent_num   = os.cpu_count() or 2
    args.render_tmp_img_filetype = "jpg"
    return args, cfg


# ── Pipeline (runs in background thread — NO st.* calls allowed here) ─────────

def run_pipeline(audio_bytes, msg_queue):
    """
    msg_queue: queue.Queue for sending (kind, message) tuples to main thread.
    Returns (vid_path, npz_path).
    """
    def log(msg, kind="run"):
        msg_queue.put(("log", kind, msg))

    log("Parsing config…")
    args, cfg = get_args_cfg()
    log("Config ready", "ok")

    # Save audio
    tmp_dir    = tempfile.mkdtemp(prefix="glsm_")
    audio_path = os.path.join(tmp_dir, "input.wav")
    with open(audio_path, "wb") as f:
        f.write(audio_bytes)
    data, sr = sf.read(audio_path)
    if data.ndim > 1:
        data = data[:, 0]
    ap = (sr, (data * 32767).astype(np.int16))

    log("Importing GestureLSM…")
    _orig = sys.argv[:]
    sys.argv = ["demo2.py", "--config", "configs/shortcut_rvqvae_128_hf.yaml"]

    # Fix Vocab pickle issue
    import __main__
    from dataloaders.build_vocab import Vocab
    __main__.Vocab = Vocab

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        import demo2 as d2
    sys.argv = _orig
    log("Import done", "ok")

    log("WhisperX alignment + dataloader…")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        trainer = d2.BaseTrainer(args, cfg, ap=ap)
    from utils import other_tools
    other_tools.load_checkpoints(trainer.model, args.test_ckpt, args.g_name)
    log("Model loaded", "ok")

    log("Inference + rendering (~3 min)…")
    result = trainer.test_demo(999)
    log("Inference done", "ok")

    # Extract paths safely
    def extract_path(r):
        if isinstance(r, str):   return r
        if isinstance(r, dict):  return r.get("value") or r.get("path") or r.get("name")
        if hasattr(r, "value"):  return r.value
        return str(r) if r else None

    render_vid = extract_path(result[0])
    npz_path   = extract_path(result[1])

    # Fallback: find latest mp4/npz in outputs
    if not render_vid or not os.path.exists(str(render_vid)):
        mp4s = sorted(glob.glob("/content/GestureLSM/outputs/audio2pose/custom/**/*.mp4",
                                recursive=True), key=os.path.getmtime)
        render_vid = mp4s[-1] if mp4s else None

    if not npz_path or not os.path.exists(str(npz_path)):
        npzs = sorted(glob.glob("/content/GestureLSM/outputs/audio2pose/custom/**/*.npz",
                                recursive=True), key=os.path.getmtime)
        npz_path = npzs[-1] if npzs else None

    # Merge audio
    if render_vid and os.path.exists(render_vid):
        wav_path  = getattr(trainer, "audio_path", None)
        final_vid = render_vid.replace(".mp4", "_final.mp4")
        if wav_path and os.path.exists(wav_path) and not os.path.exists(final_vid):
            log("Merging audio…")
            try:
                add_audio_to_video(render_vid, wav_path, final_vid)
                render_vid = final_vid
                log("Audio merged", "ok")
            except Exception as e:
                log(f"Audio merge skipped: {e}", "err")

    log("Done!", "ok")
    msg_queue.put(("result", render_vid, npz_path))


# ── UI ────────────────────────────────────────────────────────────────────────

uploaded = st.file_uploader(
    "Upload speech audio (WAV / MP3 / FLAC)", type=["wav","mp3","flac","ogg"])

# st.markdown("**Or try an example:**")
# examples = [
#     "demo/examples/2_scott_0_1_1.wav",
#     "demo/examples/2_scott_0_2_2.wav",
#     "demo/examples/2_scott_0_3_3.wav",
# ]
# cols = st.columns(len(examples))
# example_chosen = None
# for i, (col, ex) in enumerate(zip(cols, examples)):
#     with col:
#         if st.button(f"Example {i+1}", key=f"ex{i}"):
#             example_chosen = ex

st.markdown("---")
run_btn = st.button("▶  Generate Gesture Video")

log_area  = st.empty()
vid_area  = st.empty()
dl_area   = st.empty()

if run_btn:
    audio_bytes = None
    if uploaded:
        audio_bytes = uploaded.read()
    elif example_chosen:
        ex_full = os.path.join("/content/GestureLSM", example_chosen)
        if os.path.exists(ex_full):
            with open(ex_full, "rb") as f:
                audio_bytes = f.read()

    if not audio_bytes:
        st.warning("Please upload an audio file or choose an example.")
    else:
        msg_queue  = queue.Queue()
        log_lines  = []   # accumulate log HTML

        # Start pipeline in background thread
        t = threading.Thread(target=run_pipeline, args=(audio_bytes, msg_queue), daemon=True)
        t.start()

        vid_path = npz_path = None
        spinner_chars = ["⠋","⠙","⠹","⠸","⠼","⠴","⠦","⠧","⠇","⠏"]
        tick = 0

        # Main thread polls queue and updates UI
        while t.is_alive() or not msg_queue.empty():
            try:
                item = msg_queue.get(timeout=0.5)
            except queue.Empty:
                # Just update spinner
                tick += 1
                log_area.markdown(
                    "".join(log_lines) +
                    f'<div class="sbox run">{spinner_chars[tick % len(spinner_chars)]} Working…</div>',
                    unsafe_allow_html=True)
                continue

            if item[0] == "log":
                _, kind, msg = item
                log_lines.append(render_log(msg, kind))
                log_area.markdown("".join(log_lines), unsafe_allow_html=True)

            elif item[0] == "result":
                _, vid_path, npz_path = item
                break

            elif item[0] == "error":
                _, err_msg, tb = item
                st.error(err_msg)
                st.code(tb)
                st.stop()

        t.join()

        # Show result
        if vid_path and os.path.exists(str(vid_path)):
            with vid_area:
                st.video(str(vid_path))
            with dl_area:
                c1, c2 = st.columns(2)
                with c1:
                    with open(vid_path, "rb") as vf:
                        st.download_button("⬇ Download Video", vf.read(),
                                           "gesture_output.mp4", "video/mp4")
                with c2:
                    if npz_path and os.path.exists(str(npz_path)):
                        with open(npz_path, "rb") as nf:
                            st.download_button("⬇ Download Motion (.npz)",
                                               nf.read(), "motion.npz")
        else:
            st.error(f"Video not found at: {vid_path}")

Overwriting app.py


In [23]:
from pyngrok import ngrok

ngrok_key = 'abcd'
port = 8501

ngrok.set_auth_token(ngrok_key)
ngrok.connect(port).public_url

'https://marcos-isacoustic-exhortingly.ngrok-free.dev'

In [24]:
!rm -rf logs.txt && streamlit run app.py --server.port 8501 &>/content/logs.txt